# 🎯 250-Question SQL Practice Lab — E-commerce Analytics Engineering

You work as an **Analytics Engineer** at a large online retailer. The business team keeps asking questions about
sales, customers, products, payments, returns and operations. Your job is to answer them **by writing SQL yourself**.

> ✅ **Nothing in this notebook gives away an answer.** Each question has a business scenario, a task, the tables
> to use and a hint that nudges your approach without writing the query for you.

## How to use this notebook
1. Run the setup script first: `01_setup_ecommerce_lab.py` (once; it can be re-run safely).
2. Run the two *Connect to the lab* cells below.
3. Work through the questions **in order**. Each question has three cells:
   * a question cell (scenario, task, tables, difficulty),
   * a **hint** cell — try the question first; open the hint only if you are stuck,
   * an empty **SQL cell** where you write your solution and press **Shift + Enter** (or the ▶ button) to run it.
4. Track your progress in `04_sql_learning_tracker.md`. Use `03_sql_analytics_engineering_reference.md` when you need to revise a concept.

## How SQL cells work in Databricks
Every exercise cell starts with the `%sql` magic so it runs as Databricks SQL / Spark SQL regardless of the notebook's
default language. Replace the comment with your query. You can write several statements in one cell, but only
the result of the last `SELECT` is displayed — keep one query per cell.

## Where the data lives
Catalog **`workspace`**, schema (database) **`ecommerce_lab`**. After the *Connect* cell you can use plain table names such as
`customers`; otherwise write `workspace.ecommerce_lab.customers`.

**Data snapshot date: `2025-12-31`.** Orders span 2022-01-01 → 2025-12-31. Treat 2025-12-31 as "today" and
**never use `current_date()`** — otherwise your answers will drift every day.


## The data model (14 tables)

| Table | Grain (one row per…) | Key columns |
|---|---|---|
| `customers` | customer record | `customer_id` |
| `orders` | order | `order_id`, `customer_id` → customers |
| `order_items` | order line | `order_item_id`, `order_id` → orders, `product_id` → products |
| `products` | product | `product_id`, `category_id` → categories |
| `categories` | category (3-level tree) | `category_id`, `parent_category_id` → categories |
| `payments` | payment attempt | `payment_id`, `order_id` → orders |
| `returns` | returned order line | `return_id`, `order_item_id` → order_items |
| `inventory` | product × warehouse stock | `inventory_id`, `product_id`, `warehouse_id` |
| `warehouses` | warehouse | `warehouse_id` |
| `customer_activity` | clickstream event | `activity_id`, `customer_id` (NULL = anonymous), `session_id`, `product_id` |
| `order_status_history` | status change of an order | `history_id`, `order_id` |
| `customer_changes` | change-feed row for a customer (for SCD exercises) | `change_id`, `customer_id` |
| `dim_date` | calendar day | `date_key`, `full_date` |
| `orders_incoming_batch` | row in the next incremental order load | `order_id` (not unique), `batch_id` |

```
customers ──< orders ──< order_items >── products >── categories (self-referencing tree)
                │              │              │
                │              └──< returns    └──< inventory >── warehouses
                ├──< payments
                └──< order_status_history
customers ──< customer_activity >── products          customers ──< customer_changes
dim_date (calendar)                    orders_incoming_batch (staging for orders)
```

## Business glossary used in the questions
* **Valid order** — `order_status` NOT IN ('Cancelled', 'Failed'). Whenever a question says *valid*, use this rule.
* **Order statuses** — Pending, Processing, Shipped, Delivered, Cancelled, Returned, Failed.
* **Item revenue** — `order_items.line_total` (after line discount, before tax and shipping). **Order revenue** — `orders.total_amount` (items + 18 % GST + shipping).
* **Refunded amount** — `returns.refund_amount` where `return_status` = 'Refunded'.
* **Payment statuses** — Success, Failed, Pending, Refunded. **Return statuses** — Pending, Approved, Rejected, Refunded.
* All money is in **INR**.
* The data is deliberately **messy** (just like production): duplicates, NULLs, inconsistent casing, stray spaces, late-arriving rows… Early questions say *"use the value as stored / exact match"*; from Level 8 onward you clean it properly. Read each task carefully for what is expected.

## Difficulty scale
`Beginner` → `Easy` → `Intermediate` → `Advanced` → `Expert`. Difficulty rises inside every level, and each level starts harder than the previous one.


<a id="toc"></a>
## Table of contents

| # | Section | Questions | Focus |
|---|---|---|---|
| 1 | [SQL Fundamentals](#level-1) | 1–25 (25) | SELECT, WHERE, ORDER BY, LIMIT, DISTINCT, aliases, comparison and logical operators, BETWEEN, IN, LIKE, IS NULL, CASE WHEN, date filtering, calculations |
| 2 | [Aggregations and Grouping](#level-2) | 26–50 (25) | COUNT, SUM, AVG, MIN, MAX, GROUP BY, HAVING, multi-column grouping, conditional aggregation, COUNT DISTINCT |
| 3 | [Joins and Relationships](#level-3) | 51–80 (30) | INNER / LEFT / RIGHT / FULL / CROSS / SELF joins, multi-table joins, unmatched records, cardinality and fan-out |
| 4 | [Subqueries and CTEs](#level-4) | 81–105 (25) | scalar, correlated and nested subqueries, CTEs, EXISTS / NOT EXISTS, IN / NOT IN, derived tables, hierarchies |
| 5 | [Window Functions](#level-5) | 106–140 (35) | ROW_NUMBER, RANK, DENSE_RANK, LAG, LEAD, FIRST/LAST_VALUE, running totals, moving averages, frames, deduplication, top-N per group |
| 6 | [Date and Time Analytics](#level-6) | 141–160 (20) | date functions, truncation, differences, weeks / quarters, rolling windows, business days, period comparisons |
| 7 | [Advanced Analytical SQL](#level-7) | 161–190 (30) | CLV, retention, cohorts, growth, market basket, funnels, Pareto, segmentation, percentile analysis |
| 8 | [Data Cleaning and Data Quality](#level-8) | 191–210 (20) | duplicates, NULLs, COALESCE / NULLIF, string functions, regex, standardisation, integrity and consistency checks |
| 9 | [Data Warehousing and Analytics Engineering SQL](#level-9) | 211–230 (20) | grain, facts, dimensions, surrogate keys, SCD 1/2, incremental logic, late-arriving data, reconciliation, dbt-style tests |
| 10 | [Advanced Business Scenarios and Interview Challenges](#level-10) | 231–250 (20) | multi-step, multi-concept business problems |

*In Databricks, open the notebook outline (the list icon in the left sidebar) to jump straight to any section or question — every level and question title is a heading.*

## 🔌 Connect to the lab
Run these two cells once per session, then peek at the tables.

In [ ]:
# Point this notebook session at the practice schema
spark.sql("USE CATALOG workspace")
spark.sql("USE SCHEMA ecommerce_lab")
print("Connected to workspace.ecommerce_lab")

In [ ]:
%sql
SHOW TABLES IN workspace.ecommerce_lab

Handy commands while you work: `DESCRIBE TABLE orders;`, `DESCRIBE TABLE EXTENDED orders;`, `SELECT * FROM orders LIMIT 5;` — use them in your own scratch cells (add a cell with **+ Code** and start it with `%sql`).

In [ ]:
%sql
-- Scratch cell: explore the data here (not part of any question)


<a id="level-1"></a>
# Level 1: SQL Fundamentals
**Questions 1–25** · 25 questions

*Topics:* SELECT, WHERE, ORDER BY, LIMIT, DISTINCT, aliases, comparison and logical operators, BETWEEN, IN, LIKE, IS NULL, CASE WHEN, date filtering, calculations

[↑ Back to table of contents](#toc)

### Question 1: List every warehouse
**Difficulty:** Beginner &nbsp;|&nbsp; **Topic:** SELECT &nbsp;|&nbsp; **Level 1**

**Business scenario:** Operations wants a quick look at all fulfilment centres the company runs.

**Task:** Return every column and every row from the `warehouses` table.

**Tables:** `warehouses`

💡 **Hint (Question 1):** This is the simplest possible query: pick all columns from a single table. No filtering is needed.

In [ ]:
%sql
-- Question 1: write your solution here


### Question 2: Customer names and cities
**Difficulty:** Beginner &nbsp;|&nbsp; **Topic:** SELECT columns &nbsp;|&nbsp; **Level 1**

**Business scenario:** The CRM team needs a compact customer list for a mail-merge test.

**Task:** Return only `first_name`, `last_name` and `city` for every customer.

**Tables:** `customers`

💡 **Hint (Question 2):** Instead of selecting everything, list just the columns you need after SELECT, separated by commas.

In [ ]:
%sql
-- Question 2: write your solution here


### Question 3: Peek at the orders table
**Difficulty:** Beginner &nbsp;|&nbsp; **Topic:** LIMIT &nbsp;|&nbsp; **Level 1**

**Business scenario:** A new analyst wants to see what an order row looks like before writing bigger queries.

**Task:** Return `order_id`, `order_date`, `order_status` and `total_amount` for exactly 10 orders (any 10 are fine).

**Tables:** `orders`

💡 **Hint (Question 3):** LIMIT caps the number of rows returned. You do not need a WHERE clause here.

In [ ]:
%sql
-- Question 3: write your solution here


### Question 4: Find all customers from Mumbai
**Difficulty:** Beginner &nbsp;|&nbsp; **Topic:** WHERE with equality &nbsp;|&nbsp; **Level 1**

**Business scenario:** The regional marketing team in Mumbai is planning a local campaign.

**Task:** Retrieve `customer_id`, `first_name`, `last_name` and `city` for customers whose `city` is exactly 'Mumbai'. (Exact match only - messy spellings are a Level 8 topic.)

**Tables:** `customers`

💡 **Hint (Question 4):** Think about filtering rows with a comparison condition in WHERE. Text values go in single quotes.

In [ ]:
%sql
-- Question 4: write your solution here


### Question 5: Premium-priced products
**Difficulty:** Beginner &nbsp;|&nbsp; **Topic:** WHERE with numeric comparison &nbsp;|&nbsp; **Level 1**

**Business scenario:** Merchandising wants to review products priced above INR 50,000.

**Task:** Return `product_id`, `product_name` and `selling_price` for products whose `selling_price` is greater than 50000.

**Tables:** `products`

💡 **Hint (Question 5):** Numbers are not quoted. Choose the right comparison operator (strictly greater than).

In [ ]:
%sql
-- Question 5: write your solution here


### Question 6: Cheapest products first
**Difficulty:** Beginner &nbsp;|&nbsp; **Topic:** ORDER BY &nbsp;|&nbsp; **Level 1**

**Business scenario:** A pricing analyst wants the catalogue sorted from the cheapest to the most expensive item.

**Task:** Return `product_name` and `selling_price` for all products, sorted by `selling_price` ascending. Show only the first 25 rows.

**Tables:** `products`

💡 **Hint (Question 6):** ORDER BY controls the sort direction. Combine it with LIMIT to keep the output short.

In [ ]:
%sql
-- Question 6: write your solution here


### Question 7: Top 10 most expensive products
**Difficulty:** Beginner &nbsp;|&nbsp; **Topic:** ORDER BY DESC + LIMIT &nbsp;|&nbsp; **Level 1**

**Business scenario:** The brand team wants a showcase of the ten highest-priced items.

**Task:** Return `product_id`, `product_name`, `brand` and `selling_price` for the 10 products with the highest `selling_price`.

**Tables:** `products`

💡 **Hint (Question 7):** Sort in descending order first, then restrict the number of rows.

In [ ]:
%sql
-- Question 7: write your solution here


### Question 8: Which countries do customers live in?
**Difficulty:** Beginner &nbsp;|&nbsp; **Topic:** DISTINCT &nbsp;|&nbsp; **Level 1**

**Business scenario:** Finance is checking in how many countries the company has customers.

**Task:** Return the list of unique `country` values found in `customers`, sorted alphabetically.

**Tables:** `customers`

💡 **Hint (Question 8):** Apply DISTINCT to the country column so each country appears only once, then sort the result with ORDER BY.

In [ ]:
%sql
-- Question 8: write your solution here


### Question 9: Friendly column names
**Difficulty:** Beginner &nbsp;|&nbsp; **Topic:** Column aliases &nbsp;|&nbsp; **Level 1**

**Business scenario:** A dashboard developer needs business-friendly column headings for the Skincare category (`category_id` = 22).

**Task:** Return `product_name` as `Product` and `selling_price` as `Price_INR` for products in category 22.

**Tables:** `products`

💡 **Hint (Question 9):** Give columns new names with AS. Remember that category 22 is an equality filter.

In [ ]:
%sql
-- Question 9: write your solution here


### Question 10: Orders that are not delivered
**Difficulty:** Beginner &nbsp;|&nbsp; **Topic:** Inequality operator &nbsp;|&nbsp; **Level 1**

**Business scenario:** The logistics lead wants to see orders that have not reached the Delivered state.

**Task:** Return `order_id` and `order_status` for orders whose status is not 'Delivered'. Order by `order_id` and show the first 50 rows.

**Tables:** `orders`

💡 **Hint (Question 10):** Use the not-equal operator (<> or !=) on the status column.

In [ ]:
%sql
-- Question 10: write your solution here


### Question 11: Gold customers in Delhi
**Difficulty:** Beginner &nbsp;|&nbsp; **Topic:** AND &nbsp;|&nbsp; **Level 1**

**Business scenario:** The loyalty team is planning an event for Gold-tier customers living in Delhi.

**Task:** Return `customer_id`, `first_name`, `last_name` for customers where `city` = 'Delhi' AND `customer_segment` = 'Gold' (exact matches).

**Tables:** `customers`

💡 **Hint (Question 11):** Two conditions must both be true, so connect them with AND.

In [ ]:
%sql
-- Question 11: write your solution here


### Question 12: Orders paid by UPI or Wallet
**Difficulty:** Easy &nbsp;|&nbsp; **Topic:** IN / OR &nbsp;|&nbsp; **Level 1**

**Business scenario:** The payments team is studying digital-wallet style payments.

**Task:** Return `order_id`, `payment_method` and `total_amount` for orders whose `payment_method` is exactly 'UPI' or 'Wallet'. Show the first 100 rows ordered by `order_id`.

**Tables:** `orders`

💡 **Hint (Question 12):** A list of acceptable values can be written compactly with IN (...).

In [ ]:
%sql
-- Question 12: write your solution here


### Question 13: Mid-priced products
**Difficulty:** Easy &nbsp;|&nbsp; **Topic:** BETWEEN &nbsp;|&nbsp; **Level 1**

**Business scenario:** A promotion is planned for products priced from INR 1,000 to INR 5,000 inclusive.

**Task:** Return `product_id`, `product_name`, `selling_price` for products with a price in that range, cheapest first.

**Tables:** `products`

💡 **Hint (Question 13):** BETWEEN is inclusive on both ends, which matches the business wording.

In [ ]:
%sql
-- Question 13: write your solution here


### Question 14: Customers who joined in 2024
**Difficulty:** Easy &nbsp;|&nbsp; **Topic:** Date filtering &nbsp;|&nbsp; **Level 1**

**Business scenario:** Marketing is measuring the 2024 acquisition class.

**Task:** Return `customer_id`, `signup_date` and `city` for customers whose `signup_date` falls anywhere in calendar year 2024.

**Tables:** `customers`

💡 **Hint (Question 14):** signup_date is a DATE, so you can compare it to date literals such as '2024-01-01'. Think about both boundaries.

In [ ]:
%sql
-- Question 14: write your solution here


### Question 15: Products with 'Wireless' in the name
**Difficulty:** Easy &nbsp;|&nbsp; **Topic:** LIKE with wildcards &nbsp;|&nbsp; **Level 1**

**Business scenario:** The audio category manager wants every product whose name mentions 'Wireless'.

**Task:** Return `product_id` and `product_name` for products where the name contains the word Wireless (match the exact capitalisation 'Wireless').

**Tables:** `products`

💡 **Hint (Question 15):** LIKE uses % as a wildcard for any number of characters. Put it on both sides to match 'contains'.

In [ ]:
%sql
-- Question 15: write your solution here


### Question 16: Gmail customers
**Difficulty:** Easy &nbsp;|&nbsp; **Topic:** LIKE (suffix match) &nbsp;|&nbsp; **Level 1**

**Business scenario:** A partnership with Google Pay needs the customers who use Gmail addresses.

**Task:** Return `customer_id` and `email` for customers whose `email` ends with '@gmail.com' (lowercase only for now). Show the first 100 rows.

**Tables:** `customers`

💡 **Hint (Question 16):** A pattern that starts with a wildcard and ends with fixed text matches 'ends with'.

In [ ]:
%sql
-- Question 16: write your solution here


### Question 17: Customers with no email on file
**Difficulty:** Easy &nbsp;|&nbsp; **Topic:** IS NULL &nbsp;|&nbsp; **Level 1**

**Business scenario:** The CRM team cannot reach some customers by email.

**Task:** Return `customer_id`, `first_name`, `last_name` and `city` for customers whose `email` is missing (NULL).

**Tables:** `customers`

💡 **Hint (Question 17):** NULL is not a value you can compare with =. There is a dedicated predicate for it.

In [ ]:
%sql
-- Question 17: write your solution here


### Question 18: Branded but discontinued products
**Difficulty:** Easy &nbsp;|&nbsp; **Topic:** IS NOT NULL + boolean &nbsp;|&nbsp; **Level 1**

**Business scenario:** Category managers want to clean up discontinued products that still carry a brand name.

**Task:** Return `product_id`, `product_name`, `brand` for products where `brand` is not NULL and `is_active` is FALSE.

**Tables:** `products`

💡 **Hint (Question 18):** Combine IS NOT NULL with a boolean comparison using AND.

In [ ]:
%sql
-- Question 18: write your solution here


### Question 19: Margin per unit
**Difficulty:** Easy &nbsp;|&nbsp; **Topic:** Basic calculations &nbsp;|&nbsp; **Level 1**

**Business scenario:** Finance wants to know the absolute margin on each product.

**Task:** Return `product_id`, `product_name` and the margin per unit (`selling_price` minus `unit_cost`) as `margin_per_unit`. Show the 20 products with the highest margin per unit.

**Tables:** `products`

💡 **Hint (Question 19):** You can calculate with columns directly in SELECT and then sort by the alias.

In [ ]:
%sql
-- Question 19: write your solution here


### Question 20: Margin percentage for smartphones
**Difficulty:** Easy &nbsp;|&nbsp; **Topic:** Calculations + ROUND &nbsp;|&nbsp; **Level 1**

**Business scenario:** The smartphone buyer wants the gross margin percentage per model (category 41 = Smartphones).

**Task:** For products in `category_id` 41, return `product_id`, `product_name` and `margin_pct` = (selling_price - unit_cost) / selling_price * 100 rounded to 1 decimal place. Sort by `margin_pct` descending.

**Tables:** `products`

💡 **Hint (Question 20):** Mind operator order and use ROUND(expression, 1). Watch for division by the right column.

In [ ]:
%sql
-- Question 20: write your solution here


### Question 21: Orders placed on Christmas Day 2025
**Difficulty:** Easy &nbsp;|&nbsp; **Topic:** Timestamp date filtering &nbsp;|&nbsp; **Level 1**

**Business scenario:** Customer support needs every order that was placed on 25 December 2025.

**Task:** Return `order_id`, `order_date`, `order_status`, `total_amount` for orders placed on 2025-12-25 at any time of day. Remember `order_date` is a TIMESTAMP.

**Tables:** `orders`

💡 **Hint (Question 21):** A timestamp of 2025-12-25 18:30 is not equal to the date 2025-12-25. Either cast to a date or use a half-open range.

In [ ]:
%sql
-- Question 21: write your solution here


### Question 22: Large festive-season orders still in flight
**Difficulty:** Easy &nbsp;|&nbsp; **Topic:** Combining AND / OR &nbsp;|&nbsp; **Level 1**

**Business scenario:** Operations wants to chase high-value festive orders that are not yet complete.

**Task:** Return `order_id`, `order_date`, `order_status`, `total_amount` for orders placed between 2025-10-01 and 2025-11-30 (inclusive of both days) that have `total_amount` > 10000 and whose status is 'Processing' or 'Shipped'.

**Tables:** `orders`

💡 **Hint (Question 22):** Group the OR part in parentheses so it is evaluated before the ANDs.

In [ ]:
%sql
-- Question 22: write your solution here


### Question 23: Price bands for the catalogue
**Difficulty:** Easy &nbsp;|&nbsp; **Topic:** CASE WHEN &nbsp;|&nbsp; **Level 1**

**Business scenario:** The pricing team wants to label each product with a band: Budget (< 500), Mid (500-4,999), Premium (5,000-29,999), Luxury (30,000 and above).

**Task:** Return `product_id`, `selling_price` and a derived column `price_band` with those four labels. Show 50 rows ordered by `product_id`.

**Tables:** `products`

💡 **Hint (Question 23):** CASE WHEN evaluates conditions top to bottom, so order the ranges carefully.

In [ ]:
%sql
-- Question 23: write your solution here


### Question 24: Customer age bands
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** CASE WHEN + date functions &nbsp;|&nbsp; **Level 1**

**Business scenario:** Marketing wants age bands as of the data snapshot date 2025-12-31. Bands: 18-24, 25-34, 35-44, 45-54, 55+, and 'Unknown' when the date of birth is missing or implausible (age under 18 or over 100).

**Task:** Return `customer_id`, `date_of_birth`, the age in whole years at 2025-12-31 as `age`, and `age_band`. Show the first 100 customers by `customer_id`.

**Tables:** `customers`

💡 **Hint (Question 24):** Compute age from a date difference (e.g. months or days divided appropriately). Handle NULL and nonsense dates in a CASE branch before the normal bands.

In [ ]:
%sql
-- Question 24: write your solution here


### Question 25: Effective discount percentage on lines
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** Calculations on order lines &nbsp;|&nbsp; **Level 1**

**Business scenario:** The pricing team suspects heavy discounting on multi-unit lines.

**Task:** For order lines with `quantity` >= 3 and `discount` > 0, return `order_item_id`, `quantity`, `unit_price`, `discount` and `discount_pct` = discount / (quantity * unit_price) * 100 rounded to 2 decimals. Show the 25 lines with the highest `discount_pct`.

**Tables:** `order_items`

💡 **Hint (Question 25):** The discount column is an amount, not a percentage. Divide by the undiscounted line value.

In [ ]:
%sql
-- Question 25: write your solution here


<a id="level-2"></a>
# Level 2: Aggregations and Grouping
**Questions 26–50** · 25 questions

*Topics:* COUNT, SUM, AVG, MIN, MAX, GROUP BY, HAVING, multi-column grouping, conditional aggregation, COUNT DISTINCT

[↑ Back to table of contents](#toc)

### Question 26: How many customers do we have?
**Difficulty:** Easy &nbsp;|&nbsp; **Topic:** COUNT(*) &nbsp;|&nbsp; **Level 2**

**Business scenario:** The CEO asks for the size of the customer table (including any duplicate records).

**Task:** Return a single number: the total row count of `customers`.

**Tables:** `customers`

💡 **Hint (Question 26):** No filter or grouping is needed: COUNT(*) over the whole table counts every row, duplicates included.

In [ ]:
%sql
-- Question 26: write your solution here


### Question 27: Email coverage
**Difficulty:** Easy &nbsp;|&nbsp; **Topic:** COUNT(*) vs COUNT(column) &nbsp;|&nbsp; **Level 2**

**Business scenario:** CRM wants to quantify how many customer records have an email address.

**Task:** Return two columns: total customers (`total_customers`) and customers with a non-NULL email (`customers_with_email`).

**Tables:** `customers`

💡 **Hint (Question 27):** Put two counts in one SELECT: COUNT(*) counts all rows, while COUNT(column) only counts rows where that column is not NULL.

In [ ]:
%sql
-- Question 27: write your solution here


### Question 28: Total shipping collected
**Difficulty:** Easy &nbsp;|&nbsp; **Topic:** SUM &nbsp;|&nbsp; **Level 2**

**Business scenario:** Finance wants the total shipping fees recorded on all orders.

**Task:** Return the sum of `shipping_amount` across all orders as `total_shipping`.

**Tables:** `orders`

💡 **Hint (Question 28):** A simple SUM over one column with no grouping.

In [ ]:
%sql
-- Question 28: write your solution here


### Question 29: Average delivered order value
**Difficulty:** Easy &nbsp;|&nbsp; **Topic:** AVG with filter &nbsp;|&nbsp; **Level 2**

**Business scenario:** Sales wants the average order value for orders that were delivered.

**Task:** Return the average `total_amount` of orders with status 'Delivered', rounded to 2 decimals, as `avg_order_value`.

**Tables:** `orders`

💡 **Hint (Question 29):** Restrict the rows with WHERE on the status, then apply AVG and wrap it in ROUND for two decimals.

In [ ]:
%sql
-- Question 29: write your solution here


### Question 30: Order date range
**Difficulty:** Easy &nbsp;|&nbsp; **Topic:** MIN / MAX &nbsp;|&nbsp; **Level 2**

**Business scenario:** The data team needs to document the period covered by the orders table.

**Task:** Return the earliest and latest `order_date` as `first_order` and `last_order`.

**Tables:** `orders`

💡 **Hint (Question 30):** MIN and MAX work on timestamps as well as numbers.

In [ ]:
%sql
-- Question 30: write your solution here


### Question 31: Orders per status
**Difficulty:** Easy &nbsp;|&nbsp; **Topic:** GROUP BY &nbsp;|&nbsp; **Level 2**

**Business scenario:** Operations wants to see how many orders sit in each status.

**Task:** Return `order_status` and the number of orders (`order_count`), sorted by `order_count` descending.

**Tables:** `orders`

💡 **Hint (Question 31):** Every non-aggregated column in SELECT must appear in GROUP BY.

In [ ]:
%sql
-- Question 31: write your solution here


### Question 32: Customers per country
**Difficulty:** Easy &nbsp;|&nbsp; **Topic:** GROUP BY + ORDER BY &nbsp;|&nbsp; **Level 2**

**Business scenario:** International expansion wants to know where customers live.

**Task:** Return each `country` and its number of customers as `customer_count`, largest first.

**Tables:** `customers`

💡 **Hint (Question 32):** Group on the country column and count rows.

In [ ]:
%sql
-- Question 32: write your solution here


### Question 33: Revenue by payment method for delivered orders
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** GROUP BY with SUM and COUNT &nbsp;|&nbsp; **Level 2**

**Business scenario:** Finance wants to compare how much money each payment method brings in.

**Task:** For orders with status 'Delivered', return `payment_method`, number of orders and `SUM(total_amount)` as `revenue`, sorted by revenue descending. (Use the values as stored; do not try to clean the labels yet.)

**Tables:** `orders`

💡 **Hint (Question 33):** WHERE filters rows before grouping. You can compute several aggregates in one SELECT.

In [ ]:
%sql
-- Question 33: write your solution here


### Question 34: Orders per year
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** GROUP BY on date part &nbsp;|&nbsp; **Level 2**

**Business scenario:** Leadership wants to see year-on-year order volume.

**Task:** Return the calendar year of `order_date` as `order_year` and the number of orders, sorted by year.

**Tables:** `orders`

💡 **Hint (Question 34):** Use a date function to extract the year and group by that expression.

In [ ]:
%sql
-- Question 34: write your solution here


### Question 35: Cities with more than 500 customers
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** HAVING &nbsp;|&nbsp; **Level 2**

**Business scenario:** The expansion team wants to focus on large customer cities.

**Task:** Return `city` and customer count for cities with more than 500 customers (use `city` exactly as stored), largest first.

**Tables:** `customers`

💡 **Hint (Question 35):** HAVING filters groups after aggregation; WHERE cannot see aggregates.

In [ ]:
%sql
-- Question 35: write your solution here


### Question 36: Customers by country and segment
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** Multi-column GROUP BY &nbsp;|&nbsp; **Level 2**

**Business scenario:** The loyalty team needs a country x segment breakdown.

**Task:** Return `country`, `customer_segment` and customer count. Sort by country, then by count descending. (Use the stored segment values.)

**Tables:** `customers`

💡 **Hint (Question 36):** List both columns in GROUP BY. NULL segments will appear as their own group.

In [ ]:
%sql
-- Question 36: write your solution here


### Question 37: Distinct customers per order status
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** COUNT DISTINCT &nbsp;|&nbsp; **Level 2**

**Business scenario:** Support wants to know how many different customers are affected in each order status.

**Task:** Return `order_status`, the number of orders and the number of DISTINCT customers (`distinct_customers`).

**Tables:** `orders`

💡 **Hint (Question 37):** Group by status and show COUNT(*) next to COUNT(DISTINCT customer_id); a customer with many orders in one status is counted once in the second figure.

In [ ]:
%sql
-- Question 37: write your solution here


### Question 38: Best-selling products by units
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** SUM on order lines &nbsp;|&nbsp; **Level 2**

**Business scenario:** Merchandising needs the ten products with the most units sold.

**Task:** Return `product_id` and total units sold (`units_sold` = sum of `quantity`) for the 10 products with the highest units sold.

**Tables:** `order_items`

💡 **Hint (Question 38):** Group by product, sum the quantity, sort descending and limit.

In [ ]:
%sql
-- Question 38: write your solution here


### Question 39: Order status mix by year
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** Conditional aggregation &nbsp;|&nbsp; **Level 2**

**Business scenario:** Operations wants one row per year with separate columns for how many orders ended Delivered, Cancelled and Returned.

**Task:** Return `order_year`, `delivered_orders`, `cancelled_orders`, `returned_orders` (use CASE inside SUM/COUNT, or COUNT_IF) sorted by year.

**Tables:** `orders`

💡 **Hint (Question 39):** Put a CASE expression inside an aggregate to count only the rows that satisfy a condition.

In [ ]:
%sql
-- Question 39: write your solution here


### Question 40: Monthly order counts for 2025
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** Grouping by month &nbsp;|&nbsp; **Level 2**

**Business scenario:** A forecaster needs the monthly order counts for 2025.

**Task:** Return the month number and the number of orders for orders placed in 2025, in month order.

**Tables:** `orders`

💡 **Hint (Question 40):** Filter to 2025 with a date range, then group by the month extracted from the timestamp.

In [ ]:
%sql
-- Question 40: write your solution here


### Question 41: Line value by quantity
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** AVG / COUNT by group &nbsp;|&nbsp; **Level 2**

**Business scenario:** Pricing wants to see whether bigger quantities carry lower prices or bigger lines.

**Task:** For order lines with `quantity` between 1 and 5, return `quantity`, number of lines, `avg_unit_price` and `avg_line_total` (both rounded to 2 decimals), ordered by quantity.

**Tables:** `order_items`

💡 **Hint (Question 41):** One GROUP BY on quantity with several aggregates. Filter with WHERE first.

In [ ]:
%sql
-- Question 41: write your solution here


### Question 42: Products sold in more than 100 orders
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** HAVING with COUNT DISTINCT &nbsp;|&nbsp; **Level 2**

**Business scenario:** The assortment team wants products with broad order coverage rather than a few bulk buyers.

**Task:** Return `product_id` and `orders_containing` = COUNT(DISTINCT order_id) for products that appear in more than 100 different orders, highest first.

**Tables:** `order_items`

💡 **Hint (Question 42):** Use COUNT(DISTINCT ...) inside HAVING (or compute it in SELECT and filter).

In [ ]:
%sql
-- Question 42: write your solution here


### Question 43: First and last order per frequent customer
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** MIN / MAX per group &nbsp;|&nbsp; **Level 2**

**Business scenario:** CRM wants the activity window of customers who have placed at least 10 orders.

**Task:** Return `customer_id`, number of orders, first and last `order_date` for customers with 10 or more orders. Show the top 25 by number of orders.

**Tables:** `orders`

💡 **Hint (Question 43):** Group by customer, apply MIN/MAX/COUNT, and filter groups with HAVING.

In [ ]:
%sql
-- Question 43: write your solution here


### Question 44: Ten busiest days
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** GROUP BY date &nbsp;|&nbsp; **Level 2**

**Business scenario:** Capacity planning needs the days with the most orders.

**Task:** Return the calendar date of `order_date` (as `order_day`) and the number of orders for the 10 days with the most orders.

**Tables:** `orders`

💡 **Hint (Question 44):** Cast or convert the timestamp to a date before grouping, otherwise every second is its own group.

In [ ]:
%sql
-- Question 44: write your solution here


### Question 45: Payment attempt outcomes by method
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** Conditional counts &nbsp;|&nbsp; **Level 2**

**Business scenario:** The payments team wants to compare success and failure counts by method.

**Task:** Return `payment_method`, the number of attempts with `payment_status` = 'Success' (`success_count`) and = 'Failed' (`failed_count`). Use the stored values exactly; sort by `failed_count` descending.

**Tables:** `payments`

💡 **Hint (Question 45):** Two conditional aggregates side by side, grouped by payment method.

In [ ]:
%sql
-- Question 45: write your solution here


### Question 46: Cancellation rate by payment method
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Ratio via conditional aggregation &nbsp;|&nbsp; **Level 2**

**Business scenario:** Finance suspects some payment methods lead to more cancellations.

**Task:** For each `payment_method` (as stored), return total orders, cancelled orders and `cancel_rate_pct` (cancelled / total * 100, rounded to 2 decimals). Sort by `cancel_rate_pct` descending.

**Tables:** `orders`

💡 **Hint (Question 46):** Compute the numerator with a CASE inside SUM and divide by COUNT(*). Beware of integer division - multiply by 100.0.

In [ ]:
%sql
-- Question 46: write your solution here


### Question 47: Repeat cancellers
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** HAVING with conditional count &nbsp;|&nbsp; **Level 2**

**Business scenario:** Fraud prevention wants customers with at least 3 cancelled orders.

**Task:** Return `customer_id`, number of 'Cancelled' orders and total number of orders for customers with 3 or more cancelled orders, ordered by cancelled orders descending then `customer_id`.

**Tables:** `orders`

💡 **Hint (Question 47):** Count cancelled rows with a conditional aggregate and filter on it in HAVING.

In [ ]:
%sql
-- Question 47: write your solution here


### Question 48: Revenue by year and quarter
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Multi-key date grouping &nbsp;|&nbsp; **Level 2**

**Business scenario:** The CFO needs quarterly revenue for valid orders (status not 'Cancelled' and not 'Failed').

**Task:** Return `order_year`, `order_quarter`, number of orders and `SUM(total_amount)` as `revenue`, ordered chronologically.

**Tables:** `orders`

💡 **Hint (Question 48):** Extract year and quarter from the timestamp and group by both. Use NOT IN for the status exclusion.

In [ ]:
%sql
-- Question 48: write your solution here


### Question 49: Return reasons ranked
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Aggregating returns &nbsp;|&nbsp; **Level 2**

**Business scenario:** The quality team wants to know why products come back and what it costs.

**Task:** For returns with `return_status` = 'Refunded', return each `return_reason` (NULL shown as its own group), number of returns, total `refund_amount` and average `returned_quantity` (2 decimals), largest total refund first.

**Tables:** `returns`

💡 **Hint (Question 49):** Filter on status, group by reason, and use several aggregates. NULL reasons form one group automatically.

In [ ]:
%sql
-- Question 49: write your solution here


### Question 50: Weekend versus weekday sales
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Grouping by a derived flag &nbsp;|&nbsp; **Level 2**

**Business scenario:** Marketing wants to know if weekends outperform weekdays.

**Task:** Classify each valid order (status not 'Cancelled'/'Failed') as 'Weekend' (Saturday/Sunday) or 'Weekday' using `order_date`. Return the label, number of orders, total `total_amount`, and average order value (2 decimals).

**Tables:** `orders`

💡 **Hint (Question 50):** Derive the day of week with a date function, wrap it in CASE, and group by that CASE expression (or an alias of it).

In [ ]:
%sql
-- Question 50: write your solution here


<a id="level-3"></a>
# Level 3: Joins and Relationships
**Questions 51–80** · 30 questions

*Topics:* INNER / LEFT / RIGHT / FULL / CROSS / SELF joins, multi-table joins, unmatched records, cardinality and fan-out

[↑ Back to table of contents](#toc)

### Question 51: Orders with customer names
**Difficulty:** Easy &nbsp;|&nbsp; **Topic:** INNER JOIN &nbsp;|&nbsp; **Level 3**

**Business scenario:** Customer support wants to see who placed each order.

**Task:** Return `order_id`, `order_date`, `first_name` and `last_name` for the first 100 orders by `order_id`.

**Tables:** `orders`, `customers`

💡 **Hint (Question 51):** Join orders to customers on the shared customer key; INNER JOIN keeps only matching rows.

In [ ]:
%sql
-- Question 51: write your solution here


### Question 52: Products with their category
**Difficulty:** Easy &nbsp;|&nbsp; **Topic:** INNER JOIN &nbsp;|&nbsp; **Level 3**

**Business scenario:** The catalogue team needs a readable product list.

**Task:** Return `product_id`, `product_name` and the `category_name` of each product's category. Show the first 50 rows by `product_id`.

**Tables:** `products`, `categories`

💡 **Hint (Question 52):** Join on the category key that both tables share; a plain INNER JOIN works because every product has a category.

In [ ]:
%sql
-- Question 52: write your solution here


### Question 53: Stock held in the Mumbai warehouse
**Difficulty:** Easy &nbsp;|&nbsp; **Topic:** JOIN + WHERE &nbsp;|&nbsp; **Level 3**

**Business scenario:** The Mumbai site manager wants a stock list with product names.

**Task:** Return `product_name`, `stock_quantity` and `reserved_quantity` for inventory rows held in the warehouse located in the city of 'Mumbai'. Show the 30 largest `stock_quantity` values.

**Tables:** `inventory`, `warehouses`, `products`

💡 **Hint (Question 53):** Three tables: inventory is the bridge between products and warehouses.

In [ ]:
%sql
-- Question 53: write your solution here


### Question 54: Which cities generate the most orders?
**Difficulty:** Easy &nbsp;|&nbsp; **Topic:** JOIN + GROUP BY &nbsp;|&nbsp; **Level 3**

**Business scenario:** Regional heads want order volume by customer city.

**Task:** Join orders to customers and return the 10 customer `city` values (as stored) with the most orders, with the order count.

**Tables:** `orders`, `customers`

💡 **Hint (Question 54):** Join first, then group by a column from the dimension table.

In [ ]:
%sql
-- Question 54: write your solution here


### Question 55: Order line detail view
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** Three-table join &nbsp;|&nbsp; **Level 3**

**Business scenario:** Fulfilment needs a flat view of what was bought by whom.

**Task:** For orders placed on 2025-12-24 (any time of day), return `order_id`, customer `first_name`/`last_name`, `product_name`, `quantity` and `line_total`, ordered by `order_id`.

**Tables:** `orders`, `order_items`, `customers`, `products`

💡 **Hint (Question 55):** Chain joins: orders -> customers, orders -> order_items -> products. Filter the date range on the timestamp.

In [ ]:
%sql
-- Question 55: write your solution here


### Question 56: Revenue by leaf category
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** Fact-to-dimension join &nbsp;|&nbsp; **Level 3**

**Business scenario:** Category managers need revenue by the category assigned to each product.

**Task:** Return `category_name` and `SUM(line_total)` as `revenue` for order lines belonging to orders with status 'Delivered'. Show the top 10 categories.

**Tables:** `order_items`, `orders`, `products`, `categories`

💡 **Hint (Question 56):** The status filter lives on orders, the category name on categories - the join path goes through order_items and products.

In [ ]:
%sql
-- Question 56: write your solution here


### Question 57: Customers who never ordered
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** LEFT JOIN anti-pattern &nbsp;|&nbsp; **Level 3**

**Business scenario:** Marketing wants to target people who registered but never bought.

**Task:** Return `customer_id`, `first_name`, `last_name` and `signup_date` for customers with no rows in `orders`. Use a LEFT JOIN and test for NULL on the right side. Order by `customer_id` and show the first 50 rows.

**Tables:** `customers`, `orders`

💡 **Hint (Question 57):** After a LEFT JOIN, unmatched left rows have NULL in every right-side column. Filter on a right-side key being NULL.

In [ ]:
%sql
-- Question 57: write your solution here


### Question 58: Products that have never been sold
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** LEFT JOIN anti-pattern &nbsp;|&nbsp; **Level 3**

**Business scenario:** Merchandising wants to find dead catalogue items.

**Task:** Return `product_id`, `product_name`, `is_active` and `launch_date` for products that never appear in `order_items`, newest launch first. Show the first 50 rows.

**Tables:** `products`, `order_items`

💡 **Hint (Question 58):** Join from products to order_items and keep rows where the order_items side is NULL.

In [ ]:
%sql
-- Question 58: write your solution here


### Question 59: Orders per customer including zeros
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** LEFT JOIN with aggregation &nbsp;|&nbsp; **Level 3**

**Business scenario:** CRM wants order counts for the 2025 signup class, including customers who never bought.

**Task:** For customers with `signup_date` in 2025, return `customer_id` and the number of orders they have placed (0 when none). Show the 25 customers with the fewest orders, ties broken by `customer_id`.

**Tables:** `customers`, `orders`

💡 **Hint (Question 59):** COUNT(orders.order_id) counts only matched rows, so unmatched customers get 0 - unlike COUNT(*).

In [ ]:
%sql
-- Question 59: write your solution here


### Question 60: Categories with no products
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** RIGHT JOIN &nbsp;|&nbsp; **Level 3**

**Business scenario:** The taxonomy owner wants to list categories that have no product assigned directly to them (parents and mid-level nodes).

**Task:** Write the query using a RIGHT JOIN from `products` to `categories`. Return `category_id`, `category_name` and `parent_category_id` of categories with no products.

**Tables:** `products`, `categories`

💡 **Hint (Question 60):** A RIGHT JOIN keeps every row of the right-hand table. Unmatched rows have NULLs on the left.

In [ ]:
%sql
-- Question 60: write your solution here


### Question 61: Orders and payments coverage check
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** FULL OUTER JOIN &nbsp;|&nbsp; **Level 3**

**Business scenario:** Finance wants to verify two-way coverage between orders and payments.

**Task:** Using a FULL OUTER JOIN between `orders` and `payments` on `order_id`, return two numbers: `orders_without_payment` (distinct orders with no payment row) and `payments_without_order` (payment rows with no order).

**Tables:** `orders`, `payments`

💡 **Hint (Question 61):** In a full join, rows only on one side show NULL for the other side's key. Count each side separately with conditional aggregation.

In [ ]:
%sql
-- Question 61: write your solution here


### Question 62: Planning grid: warehouse x top-level category
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** CROSS JOIN &nbsp;|&nbsp; **Level 3**

**Business scenario:** The supply planning team wants a blank grid of every warehouse against every top-level category to fill with targets.

**Task:** Return `warehouse_name`, `city` and `category_name` for every combination of a warehouse and a top-level category (a category whose `parent_category_id` is NULL).

**Tables:** `warehouses`, `categories`

💡 **Hint (Question 62):** A CROSS JOIN pairs every row of one table with every row of another. Filter the category side first.

In [ ]:
%sql
-- Question 62: write your solution here


### Question 63: Category with its parent
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** SELF JOIN &nbsp;|&nbsp; **Level 3**

**Business scenario:** The BI team needs a parent-child lookup for the category tree.

**Task:** Return `category_id`, `category_name` and `parent_name` (the name of its parent category) for every category that has a parent.

**Tables:** `categories`

💡 **Hint (Question 63):** Join categories to itself using different aliases for child and parent.

In [ ]:
%sql
-- Question 63: write your solution here


### Question 64: High-value orders in India
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** Join with multiple filters &nbsp;|&nbsp; **Level 3**

**Business scenario:** Sales wants large domestic orders for a concierge programme.

**Task:** Return `order_id`, customer `first_name`, `last_name`, `city`, and `total_amount` for orders with `total_amount` > 50000 from customers whose `country` is 'India'. Sort by `total_amount` descending, show 50 rows.

**Tables:** `orders`, `customers`

💡 **Hint (Question 64):** Filters on both tables can sit in one WHERE clause after the join.

In [ ]:
%sql
-- Question 64: write your solution here


### Question 65: Sales by brand
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** Dimension attributes in aggregates &nbsp;|&nbsp; **Level 3**

**Business scenario:** Brand partnerships need a revenue league table.

**Task:** Return `brand` (NULL shown as its own row) and `SUM(line_total)` for lines of 'Delivered' orders, top 10 brands by revenue.

**Tables:** `order_items`, `orders`, `products`

💡 **Hint (Question 65):** Join items to products for the brand and to orders for the status filter.

In [ ]:
%sql
-- Question 65: write your solution here


### Question 66: How many rows does a join create?
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** Join cardinality &nbsp;|&nbsp; **Level 3**

**Business scenario:** A new analyst is worried that joining orders to order_items multiplies rows.

**Task:** Return three numbers: `order_rows` (rows in orders), `joined_rows` (rows after an INNER JOIN of orders to order_items on `order_id`) and `distinct_orders_in_join`. They should show the one-to-many effect.

**Tables:** `orders`, `order_items`

💡 **Hint (Question 66):** Count rows and distinct keys in the same SELECT after joining. One order can match many lines.

In [ ]:
%sql
-- Question 66: write your solution here


### Question 67: Payments per order
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** One-to-many relationship &nbsp;|&nbsp; **Level 3**

**Business scenario:** The payments team wants to know how many orders needed more than one payment attempt.

**Task:** Return `order_id`, number of payment rows (`attempts`) and number of rows with `payment_status` = 'Failed' (`failed_attempts`, stored value) for orders with more than 2 payment rows. Show the 20 orders with the most attempts.

**Tables:** `orders`, `payments`

💡 **Hint (Question 67):** Group payments by order_id and filter with HAVING. Orders is only needed if you want order columns.

In [ ]:
%sql
-- Question 67: write your solution here


### Question 68: Holiday versus normal days
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** Join to a date dimension &nbsp;|&nbsp; **Level 3**

**Business scenario:** Marketing wants to compare order counts on public holidays versus other days.

**Task:** Join `orders` to `dim_date` on the calendar date of `order_date`. Return `is_holiday` and the number of orders and distinct days with orders for each value.

**Tables:** `orders`, `dim_date`

💡 **Hint (Question 68):** Convert the timestamp to a DATE to match dim_date.full_date.

In [ ]:
%sql
-- Question 68: write your solution here


### Question 69: Customer revenue without double counting
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Preventing join fan-out &nbsp;|&nbsp; **Level 3**

**Business scenario:** Finance needs, per customer, the order count and item revenue - and must not inflate the order count when joining to line items.

**Task:** Return `customer_id`, `orders_placed` (distinct orders), and `item_revenue` (SUM of `line_total`) for the 20 customers with the highest `item_revenue`, considering only orders that are not 'Cancelled' or 'Failed'.

**Tables:** `orders`, `order_items`

💡 **Hint (Question 69):** COUNT(*) after a one-to-many join counts lines. Use COUNT(DISTINCT order_id) or pre-aggregate one side.

In [ ]:
%sql
-- Question 69: write your solution here


### Question 70: Revenue by country and category
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Multi-table fact/dimension join &nbsp;|&nbsp; **Level 3**

**Business scenario:** The international team wants to see which categories sell where.

**Task:** For orders with status 'Delivered', return customer `country`, product `category_name` and `SUM(line_total)` as `revenue`. Show the 15 country/category pairs with the highest revenue.

**Tables:** `order_items`, `orders`, `customers`, `products`, `categories`

💡 **Hint (Question 70):** Four joins through the fact table. Group by one column from each of two dimensions.

In [ ]:
%sql
-- Question 70: write your solution here


### Question 71: Order headers with no lines
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Anti-join &nbsp;|&nbsp; **Level 3**

**Business scenario:** Data engineering wants to know which orders lost their line items.

**Task:** Return `order_id`, `order_date`, `order_status` and `total_amount` for orders with no rows in `order_items`, ordered by `order_date`.

**Tables:** `orders`, `order_items`

💡 **Hint (Question 71):** LEFT JOIN from orders to order_items and keep rows where the item key is NULL (or use NOT EXISTS - but try the join first).

In [ ]:
%sql
-- Question 71: write your solution here


### Question 72: Delivered orders without a successful payment
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Anti-join with a filter in ON &nbsp;|&nbsp; **Level 3**

**Business scenario:** Finance suspects some delivered orders were never paid.

**Task:** Return `order_id`, `payment_method`, `total_amount` for orders with status 'Delivered' that have no payment row whose `payment_status` is 'Success' (case-insensitive after trimming). Show the 30 largest by `total_amount`.

**Tables:** `orders`, `payments`

💡 **Hint (Question 72):** Put the success condition in the ON clause (not WHERE) so unmatched orders survive the LEFT JOIN, then keep rows with NULL on the payments side.

In [ ]:
%sql
-- Question 72: write your solution here


### Question 73: Return rate by category
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Joining a sparse fact table &nbsp;|&nbsp; **Level 3**

**Business scenario:** Quality wants units returned divided by units sold for each leaf category.

**Task:** For each `category_name` return `units_sold` (SUM of `quantity`), `units_returned` (SUM of `returned_quantity`, 0 when none) and `return_rate_pct` (2 decimals). Sort by `return_rate_pct` descending. Use all order lines.

**Tables:** `order_items`, `returns`, `products`, `categories`

💡 **Hint (Question 73):** LEFT JOIN returns onto order_items so lines without returns are kept. Watch NULLs when summing.

In [ ]:
%sql
-- Question 73: write your solution here


### Question 74: Available stock by product
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Aggregating inventory across warehouses &nbsp;|&nbsp; **Level 3**

**Business scenario:** Supply planning needs the network-wide availability of each active product.

**Task:** For active products (`is_active` = TRUE), return `product_id`, `product_name`, total `stock_quantity`, total `reserved_quantity` and `available` (stock minus reserved). Show the 20 products with the lowest `available`.

**Tables:** `inventory`, `products`

💡 **Hint (Question 74):** Aggregate inventory by product before (or while) joining to products.

In [ ]:
%sql
-- Question 74: write your solution here


### Question 75: Customer records that share an email
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** SELF JOIN on a cleaned key &nbsp;|&nbsp; **Level 3**

**Business scenario:** The CRM team suspects some people registered twice.

**Task:** Self-join `customers` on `lower(trim(email))` to return pairs (`customer_id_a`, `customer_id_b`) with `customer_id_a` < `customer_id_b` that share the same non-NULL email. Show the first 50 pairs.

**Tables:** `customers`

💡 **Hint (Question 75):** Use two aliases and an inequality on the ids so each pair appears once. Normalise the email in the join condition.

In [ ]:
%sql
-- Question 75: write your solution here


### Question 76: Order, items and payments in one row
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Joining several fact tables safely &nbsp;|&nbsp; **Level 3**

**Business scenario:** Finance wants one row per order with item count, successful paid amount and number of payment attempts.

**Task:** For the 20 delivered orders with the highest `total_amount`, return `order_id`, `total_amount`, `line_count`, `units`, `paid_amount` (sum of `payment_amount` where status is 'Success' ignoring case) and `payment_attempts`. Avoid inflated sums.

**Tables:** `orders`, `order_items`, `payments`

💡 **Hint (Question 76):** Joining two child tables directly multiplies rows. Aggregate each child table to one row per order first, then join the results.

In [ ]:
%sql
-- Question 76: write your solution here


### Question 77: Browsed but never ordered
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Anti-join across behaviour and sales &nbsp;|&nbsp; **Level 3**

**Business scenario:** Growth wants customers who viewed products online but have never placed an order.

**Task:** Return `customer_id` and the number of `product_view` events for customers that have at least one 'product_view' activity and no orders. Show the 20 with the most views.

**Tables:** `customer_activity`, `orders`

💡 **Hint (Question 77):** Aggregate activity per customer, LEFT JOIN orders, and keep customers with no match.

In [ ]:
%sql
-- Question 77: write your solution here


### Question 78: Hours from order to shipment
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Self join on an event log &nbsp;|&nbsp; **Level 3**

**Business scenario:** Logistics wants to know how long it takes from placing an order to its Shipped status, by payment method.

**Task:** Join `order_status_history` to itself to pair each order's 'Pending' timestamp with its 'Shipped' timestamp. Join to `orders` for `payment_method` (as stored) and return the average hours (1 decimal) and order count per method.

**Tables:** `order_status_history`, `orders`

💡 **Hint (Question 78):** Use two aliases of the history table filtered on different statuses and joined on order_id. Subtract timestamps in seconds and convert to hours.

In [ ]:
%sql
-- Question 78: write your solution here


### Question 79: Biggest refund recipients
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Chained joins through returns &nbsp;|&nbsp; **Level 3**

**Business scenario:** Fraud and finance review the customers who received the most refunds.

**Task:** Return `customer_id`, `first_name`, `last_name`, `city` and total `refund_amount` for returns with `return_status` = 'Refunded', for the 10 customers with the highest total refund.

**Tables:** `returns`, `order_items`, `orders`, `customers`

💡 **Hint (Question 79):** Follow the key chain returns -> order_items -> orders -> customers.

In [ ]:
%sql
-- Question 79: write your solution here


### Question 80: Orders that include products from more than one warehouse region
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Join + HAVING on distinct counts &nbsp;|&nbsp; **Level 3**

**Business scenario:** Logistics wants to know which orders contain products stocked in many different warehouses (a rough split-shipment risk).

**Task:** For orders placed in December 2025, return `order_id` and the number of distinct warehouses that stock any product on that order (`warehouse_count`). Show the 15 orders with the highest count.

**Tables:** `orders`, `order_items`, `inventory`

💡 **Hint (Question 80):** orders -> order_items -> inventory (product_id). Count distinct warehouse_id per order.

In [ ]:
%sql
-- Question 80: write your solution here


<a id="level-4"></a>
# Level 4: Subqueries and CTEs
**Questions 81–105** · 25 questions

*Topics:* scalar, correlated and nested subqueries, CTEs, EXISTS / NOT EXISTS, IN / NOT IN, derived tables, hierarchies

[↑ Back to table of contents](#toc)

### Question 81: Products priced above the catalogue average
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** Scalar subquery &nbsp;|&nbsp; **Level 4**

**Business scenario:** Pricing wants to flag products priced above the average list price.

**Task:** Return `product_id`, `product_name` and `selling_price` for products with a `selling_price` greater than the average `selling_price` of all products. Show the 25 most expensive.

**Tables:** `products`

💡 **Hint (Question 81):** A subquery that returns one value can be used on the right side of a comparison.

In [ ]:
%sql
-- Question 81: write your solution here


### Question 82: Orders versus the overall average
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** Scalar subquery in SELECT &nbsp;|&nbsp; **Level 4**

**Business scenario:** Analysts want each order compared with the average value of all valid orders.

**Task:** For the 20 largest orders by `total_amount` (status not 'Cancelled'/'Failed'), return `order_id`, `total_amount`, the overall average valid-order value as `avg_valid_order` and the difference as `diff_from_avg`.

**Tables:** `orders`

💡 **Hint (Question 82):** A scalar subquery can also appear in the SELECT list.

In [ ]:
%sql
-- Question 82: write your solution here


### Question 83: Customers who paid with a wallet
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** IN subquery &nbsp;|&nbsp; **Level 4**

**Business scenario:** Partnerships wants the customers who have ever used 'Wallet' on an order.

**Task:** Return `customer_id`, `first_name`, `last_name` for customers whose `customer_id` appears in orders with `payment_method` = 'Wallet' (stored value). Show the first 50 by `customer_id`.

**Tables:** `customers`, `orders`

💡 **Hint (Question 83):** Write a subquery that returns the customer ids of Wallet orders, then keep customers whose id is IN that list.

In [ ]:
%sql
-- Question 83: write your solution here


### Question 84: Products that were never returned
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** NOT IN subquery &nbsp;|&nbsp; **Level 4**

**Business scenario:** Quality wants to know which sold products have a clean record.

**Task:** Return `product_id` for products that appear in `order_items` but none of whose lines appear in `returns`. Return the count of such products as well (`clean_products`).

**Tables:** `order_items`, `returns`

💡 **Hint (Question 84):** Find the products that DO have returns, then exclude them. Think about NULLs inside a NOT IN list.

In [ ]:
%sql
-- Question 84: write your solution here


### Question 85: Customers with at least one cancelled order
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** EXISTS &nbsp;|&nbsp; **Level 4**

**Business scenario:** Support wants a list of customers who ever cancelled an order.

**Task:** Return `customer_id`, `first_name`, `last_name` for customers for whom an order with status 'Cancelled' EXISTS. Show the first 50 by `customer_id`.

**Tables:** `customers`, `orders`

💡 **Hint (Question 85):** A correlated EXISTS subquery references the outer customer row.

In [ ]:
%sql
-- Question 85: write your solution here


### Question 86: Active customers silent in 2025
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** NOT EXISTS &nbsp;|&nbsp; **Level 4**

**Business scenario:** Retention wants customers flagged active in the CRM who placed no order in 2025.

**Task:** Return `customer_id`, `first_name`, `last_name` and `signup_date` for customers with `is_active` = TRUE who have no orders placed in calendar year 2025. Show the first 50 by `customer_id` and also the total count.

**Tables:** `customers`, `orders`

💡 **Hint (Question 86):** NOT EXISTS with the date range inside the subquery.

In [ ]:
%sql
-- Question 86: write your solution here


### Question 87: Average orders per customer
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** Derived table &nbsp;|&nbsp; **Level 4**

**Business scenario:** Leadership asks: among customers who ordered at least once, what is the typical number of orders?

**Task:** Build a derived table (subquery in FROM) with the order count per customer, then return the average (2 decimals), minimum and maximum of those counts.

**Tables:** `orders`

💡 **Hint (Question 87):** Aggregate twice: first per customer in the inner query, then across customers in the outer query.

In [ ]:
%sql
-- Question 87: write your solution here


### Question 88: Top spenders with their city
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** Single CTE &nbsp;|&nbsp; **Level 4**

**Business scenario:** Finance wants the 15 highest lifetime spenders and where they live.

**Task:** Use a CTE to compute total `total_amount` per customer over valid orders (not 'Cancelled'/'Failed'). Join the CTE to `customers` and return `customer_id`, `first_name`, `last_name`, `city` and `lifetime_spend` for the top 15.

**Tables:** `orders`, `customers`

💡 **Hint (Question 88):** WITH name AS (...) lets you reuse the per-customer totals in the main query.

In [ ]:
%sql
-- Question 88: write your solution here


### Question 89: Customers living in a warehouse city
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** IN with a subquery on another table &nbsp;|&nbsp; **Level 4**

**Business scenario:** Logistics wants to know which customers live in a city that also hosts one of our warehouses (possible same-day delivery).

**Task:** Return `customer_id`, `first_name`, `city` for customers whose `city` appears in the list of `warehouses.city` values (exact match). Also show how many such customers there are per city, largest first, in a second query.

**Tables:** `customers`, `warehouses`

💡 **Hint (Question 89):** A subquery can return a list of cities for IN. Group the filtered customers by city for the second part.

In [ ]:
%sql
-- Question 89: write your solution here


### Question 90: Orders above the customer's own average
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Correlated subquery &nbsp;|&nbsp; **Level 4**

**Business scenario:** Analysts want unusually large orders relative to each customer's own history.

**Task:** Return `order_id`, `customer_id`, `total_amount` for valid orders whose `total_amount` is more than twice that same customer's average valid order value. Show the 25 largest by `total_amount`.

**Tables:** `orders`

💡 **Hint (Question 90):** A correlated subquery recomputes the customer's average for each outer row; a CTE with a join is an alternative.

In [ ]:
%sql
-- Question 90: write your solution here


### Question 91: Latest order per customer via subquery
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Correlated subquery with MAX &nbsp;|&nbsp; **Level 4**

**Business scenario:** CRM wants each customer's most recent order.

**Task:** Return `customer_id`, `order_id`, `order_date`, `order_status` for each customer's most recent order, using a correlated subquery (or a join to a MAX subquery) rather than a window function. Show 25 rows ordered by `customer_id`.

**Tables:** `orders`

💡 **Hint (Question 91):** Compare each order date with the maximum order date for the same customer.

In [ ]:
%sql
-- Question 91: write your solution here


### Question 92: Category share of revenue
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Multiple CTEs &nbsp;|&nbsp; **Level 4**

**Business scenario:** The CFO wants each leaf category's share of total delivered item revenue.

**Task:** Use one CTE for revenue per `category_name` (lines from 'Delivered' orders) and another for the grand total. Return category, revenue and `share_pct` (2 decimals) for the top 10.

**Tables:** `order_items`, `orders`, `products`, `categories`

💡 **Hint (Question 92):** A second CTE can hold a single total that is cross-joined to the first CTE.

In [ ]:
%sql
-- Question 92: write your solution here


### Question 93: Customers who out-spend the average customer
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Nested subqueries &nbsp;|&nbsp; **Level 4**

**Business scenario:** Marketing wants customers whose total valid spend is above the average per-customer spend.

**Task:** Return `customer_id` and `total_spend` for customers whose total spend over valid orders exceeds the average of all customers' total spends. Show the 20 closest above the average (smallest `total_spend` above it).

**Tables:** `orders`

💡 **Hint (Question 93):** Per-customer totals are needed twice: once for the average and once for filtering. Use a CTE or a nested subquery.

In [ ]:
%sql
-- Question 93: write your solution here


### Question 94: Products sold in every year
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** CTE with HAVING &nbsp;|&nbsp; **Level 4**

**Business scenario:** Assortment wants the evergreen products that sold in each of 2022, 2023, 2024 and 2025.

**Task:** Return `product_id`, `product_name` for products whose order lines span all four calendar years. Return the count of such products too (`evergreen_products`).

**Tables:** `order_items`, `orders`, `products`

💡 **Hint (Question 94):** Compute distinct years per product in a CTE and compare the count with 4.

In [ ]:
%sql
-- Question 94: write your solution here


### Question 95: Net revenue per order after refunds
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Staged CTE logic &nbsp;|&nbsp; **Level 4**

**Business scenario:** Finance needs item revenue per order minus refunds actually paid.

**Task:** Stage 1: CTE of `line_total` per order. Stage 2: CTE of `refund_amount` per order for returns with status 'Refunded'. Combine to return `order_id`, `item_revenue`, `refunded` (0 if none) and `net_revenue` for the 20 orders with the highest refunds.

**Tables:** `order_items`, `returns`

💡 **Hint (Question 95):** Roll returns up to the order via order_items, then LEFT JOIN the two CTEs on order_id and COALESCE.

In [ ]:
%sql
-- Question 95: write your solution here


### Question 96: Category tree with depth
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Hierarchy / recursive-style &nbsp;|&nbsp; **Level 4**

**Business scenario:** The data team wants every category with its depth in the tree and the name of its top-level ancestor.

**Task:** Return `category_id`, `category_name`, `depth` (1 = root) and `root_category` for all 47 categories. Use a recursive CTE if your compute supports WITH RECURSIVE; otherwise chain self-joins (depth is at most 3).

**Tables:** `categories`

💡 **Hint (Question 96):** Start with root rows, then repeatedly attach children to the previous level while carrying the root name.

In [ ]:
%sql
-- Question 96: write your solution here


### Question 97: Stocked products with no recent sales
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** NOT EXISTS with a date window &nbsp;|&nbsp; **Level 4**

**Business scenario:** Inventory wants products that sit in stock but had no sale in the last 90 days up to 2025-12-31.

**Task:** Return `product_id`, `product_name` and total `stock_quantity` for active products with positive total stock and no order line on orders placed from 2025-10-02 onwards. Show the 25 with the most stock.

**Tables:** `products`, `inventory`, `order_items`, `orders`

💡 **Hint (Question 97):** Use NOT EXISTS for the sales check and aggregate inventory in a derived table or CTE.

In [ ]:
%sql
-- Question 97: write your solution here


### Question 98: Customers whose first order was in 2022 and who ordered in 2025
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Derived table with MIN/MAX &nbsp;|&nbsp; **Level 4**

**Business scenario:** Loyalty wants long-tenure buyers.

**Task:** From a derived table of per-customer first and last `order_date` (all statuses), return customers whose first order is in 2022 and last order is in 2025, with the two dates and the span in days. Show 25 ordered by span descending.

**Tables:** `orders`

💡 **Hint (Question 98):** Aggregate once per customer, then filter on the derived columns in the outer query.

In [ ]:
%sql
-- Question 98: write your solution here


### Question 99: All orders of the single biggest customer
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Scalar subquery to find an id &nbsp;|&nbsp; **Level 4**

**Business scenario:** The key-account manager needs every order of the customer with the highest valid lifetime spend.

**Task:** Use a subquery to find that customer, then return all of their orders (`order_id`, `order_date`, `order_status`, `total_amount`) ordered by date. (Ties: pick the lower `customer_id`.)

**Tables:** `orders`

💡 **Hint (Question 99):** The inner query returns one customer_id: group, sort descending and take one row.

In [ ]:
%sql
-- Question 99: write your solution here


### Question 100: Monthly revenue against the best month
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** CTE and a scalar comparison &nbsp;|&nbsp; **Level 4**

**Business scenario:** Leadership wants every month's valid revenue as a percentage of the best month ever.

**Task:** Build a monthly revenue CTE (month start date and `SUM(total_amount)` of valid orders). Return month, revenue and `pct_of_best_month` (1 decimal). Order chronologically.

**Tables:** `orders`

💡 **Hint (Question 100):** Reference MAX(revenue) from the same CTE via a scalar subquery or cross join.

In [ ]:
%sql
-- Question 100: write your solution here


### Question 101: Funnel by device
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Multiple CTEs on event data &nbsp;|&nbsp; **Level 4**

**Business scenario:** Product analytics wants, per device type, how many sessions reached each funnel step.

**Task:** Use CTEs to find sessions with a 'product_view', 'add_to_cart', 'checkout_start' and 'purchase' event. Return `device_type` (lowercased and trimmed; NULL as 'unknown'), `sessions`, and the number of sessions that reached each step.

**Tables:** `customer_activity`

💡 **Hint (Question 101):** Aggregate to session level with conditional MAX/COUNT flags, then aggregate by device.

In [ ]:
%sql
-- Question 101: write your solution here


### Question 102: Smartphone orders that also bought a case
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** EXISTS twice &nbsp;|&nbsp; **Level 4**

**Business scenario:** Cross-sell wants to know how often a smartphone order also contained a case or cover.

**Task:** Return `order_id` for orders that contain a product from `category_id` 41 (Smartphones) AND a product from `category_id` 42 (Cases & Covers), and the number of such orders (`orders_with_both`).

**Tables:** `orders`, `order_items`, `products`

💡 **Hint (Question 102):** Two EXISTS conditions with different category filters, or INTERSECT two sets of order ids.

In [ ]:
%sql
-- Question 102: write your solution here


### Question 103: Customers who returned or cancelled
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** UNION vs UNION ALL &nbsp;|&nbsp; **Level 4**

**Business scenario:** Risk wants one list of customers who either have a cancelled order or a refunded return.

**Task:** Return the number of distinct customers in each group: those with a cancelled order, those with a refunded return and those in either group (use UNION) and compare with UNION ALL row count.

**Tables:** `orders`, `returns`, `order_items`

💡 **Hint (Question 103):** UNION removes duplicates, UNION ALL keeps them. Count rows from each variant.

In [ ]:
%sql
-- Question 103: write your solution here


### Question 104: Customer spend quartile cut-offs by stages
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Staged CTEs with ranking &nbsp;|&nbsp; **Level 4**

**Business scenario:** Analytics wants a staged build: customer spend, then spend ranks, then the spend value at the 25/50/75/90th customer percentile.

**Task:** Stage 1: net valid spend per customer. Stage 2: assign `NTILE(4)` or ranks. Stage 3: return for each quartile the minimum and maximum spend and the number of customers.

**Tables:** `orders`

💡 **Hint (Question 104):** Break the logic into three named CTEs so each step is testable on its own.

In [ ]:
%sql
-- Question 104: write your solution here


### Question 105: Products that beat their category average margin
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Correlated logic vs CTE &nbsp;|&nbsp; **Level 4**

**Business scenario:** Category managers want products whose gross margin percentage is above the average margin of their own leaf category AND that sold at least 50 units.

**Task:** Return `product_id`, `product_name`, `category_name`, product margin % (from `selling_price` and `unit_cost`) and the category's average margin %. Show 25 rows ordered by the margin gap descending.

**Tables:** `products`, `categories`, `order_items`

💡 **Hint (Question 105):** Compute category averages in one CTE and units sold in another, then join both back to products.

In [ ]:
%sql
-- Question 105: write your solution here


<a id="level-5"></a>
# Level 5: Window Functions
**Questions 106–140** · 35 questions

*Topics:* ROW_NUMBER, RANK, DENSE_RANK, LAG, LEAD, FIRST/LAST_VALUE, running totals, moving averages, frames, deduplication, top-N per group

[↑ Back to table of contents](#toc)

### Question 106: Number each customer's orders
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** ROW_NUMBER &nbsp;|&nbsp; **Level 5**

**Business scenario:** CRM wants to know which order was a customer's 1st, 2nd, 3rd...

**Task:** For customers with `customer_id` between 1 and 20, return `customer_id`, `order_id`, `order_date` and `order_seq` (1 for the earliest order). Order by customer then sequence.

**Tables:** `orders`

💡 **Hint (Question 106):** ROW_NUMBER() with PARTITION BY the customer and ORDER BY the order date.

In [ ]:
%sql
-- Question 106: write your solution here


### Question 107: Each customer's latest order
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** ROW_NUMBER for 'latest per group' &nbsp;|&nbsp; **Level 5**

**Business scenario:** A CRM export needs exactly one row per customer: their most recent order.

**Task:** Return `customer_id`, `order_id`, `order_date`, `order_status`, `total_amount` for each customer's latest order (all statuses). Show 25 rows ordered by `customer_id`. If two orders share a timestamp, prefer the higher `order_id`.

**Tables:** `orders`

💡 **Hint (Question 107):** Number orders per customer from newest to oldest, then keep sequence 1 in an outer query or QUALIFY clause.

In [ ]:
%sql
-- Question 107: write your solution here


### Question 108: Top 10 products by units, with ties
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** RANK &nbsp;|&nbsp; **Level 5**

**Business scenario:** Merchandising wants a top-10 list that keeps tied products together.

**Task:** Rank products by total units sold (sum of `quantity`) using RANK. Return `product_id`, `units_sold`, `unit_rank` for every product whose rank is 10 or better.

**Tables:** `order_items`

💡 **Hint (Question 108):** Aggregate first, then apply RANK() OVER (ORDER BY units_sold DESC) and filter in an outer query.

In [ ]:
%sql
-- Question 108: write your solution here


### Question 109: Rank cities by customer count
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** DENSE_RANK &nbsp;|&nbsp; **Level 5**

**Business scenario:** The expansion team wants a city ranking with no gaps in rank numbers.

**Task:** Return `city` (as stored), `customer_count` and `city_rank` using DENSE_RANK on customer count descending. Show cities with rank 1 to 8.

**Tables:** `customers`

💡 **Hint (Question 109):** Count customers per city first, then rank those counts with DENSE_RANK (no gaps after ties) and filter the rank in an outer query.

In [ ]:
%sql
-- Question 109: write your solution here


### Question 110: RANK versus DENSE_RANK versus ROW_NUMBER
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** Ranking functions compared &nbsp;|&nbsp; **Level 5**

**Business scenario:** A trainee wants to see how the three numbering functions treat ties.

**Task:** Aggregate units sold per product. For the 20 best sellers return `product_id`, `units_sold`, and three columns: `row_num`, `rnk`, `dense_rnk` computed on units sold descending (break ROW_NUMBER ties by `product_id`).

**Tables:** `order_items`

💡 **Hint (Question 110):** Define all three window functions with the same ORDER BY and compare the output where units tie.

In [ ]:
%sql
-- Question 110: write your solution here


### Question 111: Each order's share of the customer's spend
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** SUM() OVER (PARTITION BY) &nbsp;|&nbsp; **Level 5**

**Business scenario:** CRM wants to know how much each order contributes to that customer's total.

**Task:** For customers 1 to 10 and valid orders (not 'Cancelled'/'Failed'), return `customer_id`, `order_id`, `total_amount`, `customer_total` (window sum) and `pct_of_customer_total` (1 decimal).

**Tables:** `orders`

💡 **Hint (Question 111):** A window SUM over PARTITION BY customer_id repeats the customer total on every row without collapsing rows.

In [ ]:
%sql
-- Question 111: write your solution here


### Question 112: Running revenue for December 2025
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** Running total &nbsp;|&nbsp; **Level 5**

**Business scenario:** Finance wants a cumulative daily revenue line for the last month.

**Task:** Aggregate valid-order `total_amount` by calendar day for 2025-12-01 to 2025-12-31 and return `order_day`, `daily_revenue` and `cumulative_revenue`.

**Tables:** `orders`

💡 **Hint (Question 112):** Aggregate by day in a CTE, then SUM() OVER (ORDER BY day) with a running frame.

In [ ]:
%sql
-- Question 112: write your solution here


### Question 113: Cumulative order count per customer
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** COUNT() OVER &nbsp;|&nbsp; **Level 5**

**Business scenario:** CRM wants a counter showing the n-th order the customer has placed.

**Task:** For customers 1 to 15, return `customer_id`, `order_id`, `order_date` and `orders_so_far` (a running COUNT of that customer's orders up to and including this one).

**Tables:** `orders`

💡 **Hint (Question 113):** COUNT(*) OVER (PARTITION BY ... ORDER BY ...) gives a cumulative count.

In [ ]:
%sql
-- Question 113: write your solution here


### Question 114: Price versus category average
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** AVG() OVER &nbsp;|&nbsp; **Level 5**

**Business scenario:** Pricing wants each product compared with the average price of its own category.

**Task:** Return `product_id`, `product_name`, `category_id`, `selling_price`, `category_avg_price` (2 decimals) and `diff_from_avg` for products in categories 41 to 44. Order by category then difference descending.

**Tables:** `products`

💡 **Hint (Question 114):** AVG(selling_price) OVER (PARTITION BY category_id) adds the category average to each row.

In [ ]:
%sql
-- Question 114: write your solution here


### Question 115: Days between consecutive orders
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** LAG &nbsp;|&nbsp; **Level 5**

**Business scenario:** Retention wants the gap between a customer's orders.

**Task:** For customers 1 to 20 (all orders), return `customer_id`, `order_id`, `order_date`, `previous_order_date` and `days_since_previous` (NULL for the first order).

**Tables:** `orders`

💡 **Hint (Question 115):** LAG(order_date) OVER (PARTITION BY customer_id ORDER BY order_date), then a date difference.

In [ ]:
%sql
-- Question 115: write your solution here


### Question 116: Next order and the wait for it
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** LEAD &nbsp;|&nbsp; **Level 5**

**Business scenario:** CRM wants to see the date of the next order and whether a customer has not ordered again.

**Task:** For customers 1 to 20, return `customer_id`, `order_id`, `order_date`, `next_order_date`, `days_to_next_order` and a flag `is_last_order` that is TRUE when there is no next order.

**Tables:** `orders`

💡 **Hint (Question 116):** LEAD looks forward. A NULL result from LEAD marks the final order.

In [ ]:
%sql
-- Question 116: write your solution here


### Question 117: Compare every order with the customer's first order
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** FIRST_VALUE &nbsp;|&nbsp; **Level 5**

**Business scenario:** Analysts want to see how much an order differs from the customer's first order value.

**Task:** For customers 1 to 10 (valid orders), return `customer_id`, `order_id`, `order_date`, `total_amount`, `first_order_amount` and `change_vs_first`.

**Tables:** `orders`

💡 **Hint (Question 117):** FIRST_VALUE(total_amount) OVER (PARTITION BY ... ORDER BY order_date) repeats the first value on each row.

In [ ]:
%sql
-- Question 117: write your solution here


### Question 118: Last order status per customer
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** LAST_VALUE with an explicit frame &nbsp;|&nbsp; **Level 5**

**Business scenario:** Support wants each customer's most recent order status on every row of their history.

**Task:** For customers 1 to 10, return `customer_id`, `order_id`, `order_date`, `order_status` and `latest_status` computed with LAST_VALUE. Make sure the window frame really reaches the last row.

**Tables:** `orders`

💡 **Hint (Question 118):** With ORDER BY, the default frame stops at the current row, so LAST_VALUE returns the current value. Specify ROWS BETWEEN UNBOUNDED PRECEDING AND UNBOUNDED FOLLOWING.

In [ ]:
%sql
-- Question 118: write your solution here


### Question 119: 7-day moving average of daily orders
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** ROWS BETWEEN moving average &nbsp;|&nbsp; **Level 5**

**Business scenario:** Operations wants a smoothed daily order trend for the last quarter.

**Task:** Aggregate order counts per calendar day for 2025-10-01 to 2025-12-31. Return `order_day`, `orders`, and `moving_avg_7d` (average of the current day and 6 previous days, 1 decimal).

**Tables:** `orders`

💡 **Hint (Question 119):** Aggregate to one row per day first; then AVG(...) OVER (ORDER BY day ROWS BETWEEN 6 PRECEDING AND CURRENT ROW).

In [ ]:
%sql
-- Question 119: write your solution here


### Question 120: 3-month moving average revenue
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Moving average on months &nbsp;|&nbsp; **Level 5**

**Business scenario:** The CFO wants a smoothed monthly revenue series for valid orders.

**Task:** Return month start, `monthly_revenue`, `moving_avg_3m` (current and 2 previous months) and `months_in_window` (how many months the average really covers) for all months from 2022 to 2025.

**Tables:** `orders`

💡 **Hint (Question 120):** Use a monthly CTE and a ROWS frame; the first two months only have a partial window.

In [ ]:
%sql
-- Question 120: write your solution here


### Question 121: Average of the previous three orders
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** ROWS BETWEEN excluding current row &nbsp;|&nbsp; **Level 5**

**Business scenario:** Fraud wants to compare each order with the customer's recent average, excluding the current order.

**Task:** For valid orders of customers 1 to 30, return `customer_id`, `order_id`, `total_amount`, `avg_prev_3_orders` (average of up to the 3 prior orders, NULL if none) and the ratio `amount_vs_recent_avg`.

**Tables:** `orders`

💡 **Hint (Question 121):** Frame: ROWS BETWEEN 3 PRECEDING AND 1 PRECEDING.

In [ ]:
%sql
-- Question 121: write your solution here


### Question 122: Trailing 7-day revenue with RANGE
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** RANGE BETWEEN &nbsp;|&nbsp; **Level 5**

**Business scenario:** Finance wants a trailing 7-calendar-day revenue even when some days have no row.

**Task:** Aggregate valid revenue per calendar day for 2025. Return `order_day`, `daily_revenue` and `trailing_7d_revenue` using a RANGE frame over the day (a numeric day number or a date/interval frame) covering the current day and the 6 previous calendar days.

**Tables:** `orders`

💡 **Hint (Question 122):** RANGE works on the ORDER BY value rather than row positions. Convert the day to a number (e.g. days since a fixed date) if an interval frame is not accepted.

In [ ]:
%sql
-- Question 122: write your solution here


### Question 123: Duplicate customer records: which to keep?
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Deduplication with ROW_NUMBER &nbsp;|&nbsp; **Level 5**

**Business scenario:** Data stewardship needs a keep/discard decision for customers sharing the same email.

**Task:** For customers with a non-NULL email, partition by `lower(trim(email))` and keep the record with the earliest `signup_date` (tie: lowest `customer_id`). Return `customer_id`, normalised email, `signup_date`, `rn` and a flag `keep` (1 for rn = 1) for emails that occur more than once. Show 40 rows.

**Tables:** `customers`

💡 **Hint (Question 123):** ROW_NUMBER() OVER (PARTITION BY normalised_email ORDER BY signup_date, customer_id); use COUNT() OVER to restrict to duplicated emails.

In [ ]:
%sql
-- Question 123: write your solution here


### Question 124: Duplicate gateway callbacks
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Deduplication of payments &nbsp;|&nbsp; **Level 5**

**Business scenario:** Finance found repeated payment rows with the same gateway reference.

**Task:** Among payment rows with `payment_status` in any casing of 'success', find references that appear more than once. Return `transaction_reference`, `payment_id`, `order_id`, `payment_date`, `rn` (1 = earliest) and the total number of extra rows that would be removed by keeping only rn = 1 (`rows_to_remove`, as a separate summary).

**Tables:** `payments`

💡 **Hint (Question 124):** Number rows per transaction_reference by payment_date. Rows with rn > 1 are the removable duplicates.

In [ ]:
%sql
-- Question 124: write your solution here


### Question 125: Top 3 products per category by revenue
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Top N per group &nbsp;|&nbsp; **Level 5**

**Business scenario:** Category managers want their three best products by item revenue.

**Task:** Rank products within each leaf `category_name` by total `line_total` (all orders not 'Cancelled'/'Failed'). Return the top 3 per category with `category_name`, `product_id`, `product_name`, `revenue`, `rank_in_category`. Order by category then rank.

**Tables:** `order_items`, `orders`, `products`, `categories`

💡 **Hint (Question 125):** Aggregate by product first. Then DENSE_RANK or ROW_NUMBER with PARTITION BY category and filter rank <= 3.

In [ ]:
%sql
-- Question 125: write your solution here


### Question 126: Top 3 spenders in each city
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Top N per group &nbsp;|&nbsp; **Level 5**

**Business scenario:** Regional managers want their top three highest-spending customers.

**Task:** Using valid orders, return for each customer `city` (as stored) the three customers with the highest total `total_amount`: `city`, `customer_id`, `first_name`, `last_name`, `total_spend`, `spend_rank`. Restrict to cities 'Mumbai', 'Delhi' and 'Bengaluru' to keep the output short.

**Tables:** `orders`, `customers`

💡 **Hint (Question 126):** Aggregate spend per customer, then rank within each city and filter.

In [ ]:
%sql
-- Question 126: write your solution here


### Question 127: Customer spend quartiles
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** NTILE &nbsp;|&nbsp; **Level 5**

**Business scenario:** Marketing wants to split customers into four equal spend groups.

**Task:** Compute total valid spend per customer (customers with at least one valid order). Assign `NTILE(4)` by spend descending and return `spend_quartile`, customer count, `min_spend`, `max_spend` and `avg_spend` per quartile.

**Tables:** `orders`

💡 **Hint (Question 127):** NTILE needs the per-customer totals computed first. Group the result by the tile number.

In [ ]:
%sql
-- Question 127: write your solution here


### Question 128: Price percentile within the category
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** PERCENT_RANK / CUME_DIST &nbsp;|&nbsp; **Level 5**

**Business scenario:** Pricing wants to know how a product's price ranks within its leaf category.

**Task:** For products in categories 41 (Smartphones) and 45 (Laptops), return `product_id`, `category_id`, `selling_price`, `price_percent_rank` and `price_cume_dist` (both rounded to 3 decimals) computed within category.

**Tables:** `products`

💡 **Hint (Question 128):** Two window functions over PARTITION BY category_id ORDER BY selling_price.

In [ ]:
%sql
-- Question 128: write your solution here


### Question 129: Order status transitions
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** LAG on an event log &nbsp;|&nbsp; **Level 5**

**Business scenario:** Process analysts want to know how often each status transition occurs.

**Task:** From `order_status_history`, use LAG to find each row's previous status within the order. Return `previous_status`, `order_status` and the number of transitions, excluding each order's first row. Sort by count descending.

**Tables:** `order_status_history`

💡 **Hint (Question 129):** LAG(order_status) OVER (PARTITION BY order_id ORDER BY status_timestamp).

In [ ]:
%sql
-- Question 129: write your solution here


### Question 130: First and last purchase per customer
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** First/last transactions &nbsp;|&nbsp; **Level 5**

**Business scenario:** The lifecycle team wants first and last order details side by side.

**Task:** For customers with at least one valid order, return `customer_id`, `first_order_date`, `first_order_amount`, `last_order_date`, `last_order_amount`, and `orders` count. One row per customer, show 25 ordered by `customer_id`.

**Tables:** `orders`

💡 **Hint (Question 130):** Use FIRST_VALUE and LAST_VALUE (with a full frame) or two ROW_NUMBER directions, then collapse to one row per customer.

In [ ]:
%sql
-- Question 130: write your solution here


### Question 131: Month-over-month revenue growth
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** LAG on aggregated data &nbsp;|&nbsp; **Level 5**

**Business scenario:** Leadership wants month-on-month revenue growth for 2025.

**Task:** Aggregate valid revenue per month, then return `month_start`, `revenue`, `prev_month_revenue` and `mom_growth_pct` (1 decimal) for the months of 2025 (January compares with December 2024).

**Tables:** `orders`

💡 **Hint (Question 131):** Build all months first (including Dec 2024), apply LAG, and filter to 2025 afterwards - otherwise January loses its previous month.

In [ ]:
%sql
-- Question 131: write your solution here


### Question 132: Running share of revenue by city
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Cumulative percentage &nbsp;|&nbsp; **Level 5**

**Business scenario:** The regional head wants to see how quickly cities add up to total revenue.

**Task:** Rank customer cities (as stored) by valid revenue descending. Return `city`, `revenue`, `pct_of_total` and `cumulative_pct` (running share, 1 decimal).

**Tables:** `orders`, `customers`

💡 **Hint (Question 132):** SUM(revenue) OVER () for the total and a running SUM(...) OVER (ORDER BY revenue DESC) for the cumulative part.

In [ ]:
%sql
-- Question 132: write your solution here


### Question 133: Longest gap between orders
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** LAG + aggregation &nbsp;|&nbsp; **Level 5**

**Business scenario:** Retention wants the largest silence each customer had between two orders.

**Task:** For customers with at least 3 orders, return `customer_id`, number of orders and `max_gap_days` (the longest gap between consecutive orders). Show the 20 customers with the largest gap.

**Tables:** `orders`

💡 **Hint (Question 133):** Compute day gaps with LAG in a CTE, then take MAX per customer.

In [ ]:
%sql
-- Question 133: write your solution here


### Question 134: Reactivation after a long gap
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** LAG + CASE &nbsp;|&nbsp; **Level 5**

**Business scenario:** Retention wants to know when customers came back after at least 180 days of silence.

**Task:** For each valid order whose gap since the same customer's previous valid order is 180 days or more, return `customer_id`, `order_id`, `previous_order_date`, `order_date`, `gap_days`. Show the 25 longest gaps.

**Tables:** `orders`

💡 **Hint (Question 134):** LAG over valid orders only, filter on the gap in an outer query.

In [ ]:
%sql
-- Question 134: write your solution here


### Question 135: Cumulative new customers by month
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Window over a first-event CTE &nbsp;|&nbsp; **Level 5**

**Business scenario:** Growth wants the number of customers who placed their first valid order in each month, with a running total.

**Task:** Determine each customer's first valid order date, count customers by first-order month and return `month_start`, `new_customers`, `cumulative_customers`.

**Tables:** `orders`

💡 **Hint (Question 135):** First-order date is a per-customer aggregate (or ROW_NUMBER = 1). Then group by month and apply a running SUM.

In [ ]:
%sql
-- Question 135: write your solution here


### Question 136: What happens after add-to-cart?
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** LEAD on event data &nbsp;|&nbsp; **Level 5**

**Business scenario:** Product wants to know what users do immediately after adding something to the cart.

**Task:** Within each session, order events by `activity_timestamp` (tie-break by `activity_id`). For rows with `activity_type` = 'add_to_cart', return the next activity type (`next_activity`; use 'session_end' when none) and its count and percentage.

**Tables:** `customer_activity`

💡 **Hint (Question 136):** LEAD(activity_type) OVER (PARTITION BY session_id ORDER BY activity_timestamp, activity_id); filter after computing.

In [ ]:
%sql
-- Question 136: write your solution here


### Question 137: Product rank movement month over month
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** RANK + LAG &nbsp;|&nbsp; **Level 5**

**Business scenario:** Merchandising wants to see how products move up and down the monthly revenue ranking.

**Task:** For each month of 2025, rank products by item revenue (valid orders). Return the top 10 products per month with `month_start`, `product_id`, `revenue`, `rank_this_month`, `rank_prev_month` (NULL if not ranked or no data) and `rank_change`.

**Tables:** `order_items`, `orders`

💡 **Hint (Question 137):** Rank per month in one CTE, then LAG the rank by product ordered by month. Rank all products before filtering to the top 10.

In [ ]:
%sql
-- Question 137: write your solution here


### Question 138: Time between events in a session
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** LAG on timestamps &nbsp;|&nbsp; **Level 5**

**Business scenario:** UX research wants to know how long users spend before each type of action.

**Task:** Within each session, compute seconds since the previous event. Return `activity_type`, number of events with a previous event, `avg_seconds_since_previous` (1 decimal) and `max_seconds_since_previous`. Sort by the average descending.

**Tables:** `customer_activity`

💡 **Hint (Question 138):** Subtract the LAG(timestamp) from the current timestamp using unix_timestamp or timestamp difference.

In [ ]:
%sql
-- Question 138: write your solution here


### Question 139: Orders higher than the previous order, twice in a row
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** LAG twice &nbsp;|&nbsp; **Level 5**

**Business scenario:** Marketing wants customers whose order value rose on two consecutive orders.

**Task:** For valid orders, find customers who have an order whose amount is greater than the previous order, which in turn was greater than the one before it. Return `customer_id`, `order_id` of the third order and the three amounts.

**Tables:** `orders`

💡 **Hint (Question 139):** Use LAG with offsets 1 and 2 in the same window definition.

In [ ]:
%sql
-- Question 139: write your solution here


### Question 140: Median order value per month
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Percentiles by group &nbsp;|&nbsp; **Level 5**

**Business scenario:** The CFO prefers the median to the mean because of outlier orders.

**Task:** For each month of 2025, return `month_start`, average valid order value and median valid order value (use `percentile_approx` or `percentile`), both 2 decimals, plus the number of orders.

**Tables:** `orders`

💡 **Hint (Question 140):** Aggregate percentile functions are used with GROUP BY; no window is required.

In [ ]:
%sql
-- Question 140: write your solution here


<a id="level-6"></a>
# Level 6: Date and Time Analytics
**Questions 141–160** · 20 questions

*Topics:* date functions, truncation, differences, weeks / quarters, rolling windows, business days, period comparisons

[↑ Back to table of contents](#toc)

### Question 141: Break a timestamp into parts
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** Date part functions &nbsp;|&nbsp; **Level 6**

**Business scenario:** A BI tool needs separate columns for date parts.

**Task:** For the first 20 orders by `order_id`, return `order_id`, the year, month, day of month, day name (e.g. Monday) and hour of `order_date`.

**Tables:** `orders`

💡 **Hint (Question 141):** Use year(), month(), day(), date_format() with 'EEEE' and hour().

In [ ]:
%sql
-- Question 141: write your solution here


### Question 142: Orders by weekday
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** Day-of-week analysis &nbsp;|&nbsp; **Level 6**

**Business scenario:** Operations wants staffing guidance by weekday.

**Task:** Return the weekday name, number of orders and average order value (2 decimals) for valid orders, sorted Monday to Sunday.

**Tables:** `orders`

💡 **Hint (Question 142):** You need a numeric weekday for ordering and a text weekday for display; group by both.

In [ ]:
%sql
-- Question 142: write your solution here


### Question 143: Orders by hour of day
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** Time-of-day distribution &nbsp;|&nbsp; **Level 6**

**Business scenario:** Marketing wants the best hours for push notifications.

**Task:** Return `order_hour` (0-23) and the number of orders placed in each hour, plus the percentage of all orders (1 decimal).

**Tables:** `orders`

💡 **Hint (Question 143):** hour(order_date) grouped; use a window SUM() OVER () for the percentage.

In [ ]:
%sql
-- Question 143: write your solution here


### Question 144: Monthly revenue with date_trunc
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** Date truncation &nbsp;|&nbsp; **Level 6**

**Business scenario:** Finance wants a monthly revenue series for 2025.

**Task:** Return `month_start` (first day of the month as a date) and valid revenue for each month of 2025, in chronological order.

**Tables:** `orders`

💡 **Hint (Question 144):** date_trunc('MONTH', ...) normalises dates to the first of the month; cast back to DATE if needed.

In [ ]:
%sql
-- Question 144: write your solution here


### Question 145: Days from order to last status update for delivered orders
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** Date differences &nbsp;|&nbsp; **Level 6**

**Business scenario:** Logistics wants a proxy for delivery time using the last update on delivered orders.

**Task:** For orders with status 'Delivered', return `payment_method` (as stored), the number of orders and the average number of days between `order_date` and `updated_at` (1 decimal).

**Tables:** `orders`

💡 **Hint (Question 145):** Subtract the dates (or use datediff on the date parts) and average per group.

In [ ]:
%sql
-- Question 145: write your solution here


### Question 146: Customer tenure at the snapshot date
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** Date arithmetic &nbsp;|&nbsp; **Level 6**

**Business scenario:** CRM wants tenure in days and months as of 2025-12-31.

**Task:** For the first 25 customers by `customer_id`, return `signup_date`, `tenure_days` and `tenure_months` (whole months) at 2025-12-31.

**Tables:** `customers`

💡 **Hint (Question 146):** datediff for days and months_between (floored) for months.

In [ ]:
%sql
-- Question 146: write your solution here


### Question 147: Orders in the final 30 days of data
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** Relative date windows &nbsp;|&nbsp; **Level 6**

**Business scenario:** Operations wants a summary of the last 30 days including 2025-12-31 (so from 2025-12-02).

**Task:** Return the number of orders, number of distinct customers and total `total_amount` of valid orders placed in that window.

**Tables:** `orders`

💡 **Hint (Question 147):** Use date arithmetic against the fixed snapshot date - never current_date() - and mind the timestamp upper bound.

In [ ]:
%sql
-- Question 147: write your solution here


### Question 148: Quarterly orders: 2024 against 2025
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** Quarter analysis &nbsp;|&nbsp; **Level 6**

**Business scenario:** Leadership wants the four quarters of 2024 and 2025 side by side.

**Task:** Return one row per quarter number (1-4) with `orders_2024`, `orders_2025` and `growth_pct` (1 decimal).

**Tables:** `orders`

💡 **Hint (Question 148):** Group by the quarter and use a CASE on the year inside COUNT or SUM to build the 2024 and 2025 columns.

In [ ]:
%sql
-- Question 148: write your solution here


### Question 149: Weekly orders in November 2025
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** Week truncation &nbsp;|&nbsp; **Level 6**

**Business scenario:** Marketing wants weekly volumes for the festive campaign.

**Task:** Return the Monday-start week (`week_start`) and the number of orders for orders placed in November 2025.

**Tables:** `orders`

💡 **Hint (Question 149):** date_trunc('WEEK', ...) starts weeks on Monday in Spark/Databricks.

In [ ]:
%sql
-- Question 149: write your solution here


### Question 150: Signup cohorts older than 24 months
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** months_between / add_months &nbsp;|&nbsp; **Level 6**

**Business scenario:** Loyalty wants to know how many customers have been with us for more than 24 months at 2025-12-31, by signup year.

**Task:** Return `signup_year` and the number of customers whose `signup_date` is at least 24 months before 2025-12-31.

**Tables:** `customers`

💡 **Hint (Question 150):** Subtract 24 months from 2025-12-31 with add_months to get a cut-off date, filter signup_date against it and group by the signup year.

In [ ]:
%sql
-- Question 150: write your solution here


### Question 151: Same month last year
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Period comparison with a self-join &nbsp;|&nbsp; **Level 6**

**Business scenario:** The CFO wants this month's revenue next to the same month a year earlier.

**Task:** Aggregate valid monthly revenue and return `month_start`, `revenue`, `revenue_same_month_last_year` and `yoy_growth_pct` for 2024-01 through 2025-12. Use a self-join on months shifted by 12 (or add_months).

**Tables:** `orders`

💡 **Hint (Question 151):** Join the monthly CTE to itself on prior.month_start = add_months(current.month_start, -12).

In [ ]:
%sql
-- Question 151: write your solution here


### Question 152: Business days to refund
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Business-day calculation with a calendar table &nbsp;|&nbsp; **Level 6**

**Business scenario:** The returns team wants the number of business days (Monday-Friday, excluding holidays) between the order date and the return date.

**Task:** For returns with `return_status` = 'Refunded' raised in 2025, return the average number of business days from order to return date (1 decimal), plus the 90th percentile. Use `dim_date` to count qualifying days.

**Tables:** `returns`, `order_items`, `orders`, `dim_date`

💡 **Hint (Question 152):** Count dim_date rows between the two dates where is_weekend is false and is_holiday is false.

In [ ]:
%sql
-- Question 152: write your solution here


### Question 153: Daily orders with zero-filled gaps
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Date spine &nbsp;|&nbsp; **Level 6**

**Business scenario:** A BI tool needs a row for every day of 2025, including days with no order.

**Task:** Build a date spine for 2025 from `dim_date`, LEFT JOIN daily order counts and return `calendar_date`, `orders` (0 if none) and `rolling_7d_orders`.

**Tables:** `dim_date`, `orders`

💡 **Hint (Question 153):** Calendar first, facts second: LEFT JOIN from dim_date and COALESCE the counts.

In [ ]:
%sql
-- Question 153: write your solution here


### Question 154: Fiscal year and quarter revenue
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Fiscal calendar &nbsp;|&nbsp; **Level 6**

**Business scenario:** Finance reports on an April-March fiscal year.

**Task:** Using `dim_date`, return `fiscal_year`, `fiscal_quarter`, number of valid orders and revenue for all fiscal quarters that have data, ordered chronologically.

**Tables:** `orders`, `dim_date`

💡 **Hint (Question 154):** Join on the date and group by the fiscal columns from the calendar.

In [ ]:
%sql
-- Question 154: write your solution here


### Question 155: How late do orders arrive in the warehouse?
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Timestamp differences &nbsp;|&nbsp; **Level 6**

**Business scenario:** Data engineering wants to quantify late-arriving order records.

**Task:** For each month of 2025 (by `order_date`), return the number of orders, the number whose `loaded_at` is more than 48 hours after `updated_at` and the late percentage (2 decimals).

**Tables:** `orders`

💡 **Hint (Question 155):** Convert timestamps to seconds and compare against 48*3600 (or use interval arithmetic).

In [ ]:
%sql
-- Question 155: write your solution here


### Question 156: Activation within 7 days of signup
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Time-to-event by cohort &nbsp;|&nbsp; **Level 6**

**Business scenario:** Growth wants the share of each monthly signup cohort that placed a valid order within 7 days of signing up.

**Task:** Return `signup_month`, cohort size, customers with a valid order within 7 days of `signup_date` and `activation_rate_pct` (1 decimal), for signup months in 2024.

**Tables:** `customers`, `orders`

💡 **Hint (Question 156):** Join customers to orders on customer, filter on the date difference, and count distinct customers.

In [ ]:
%sql
-- Question 156: write your solution here


### Question 157: Holiday effect on daily orders
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Calendar attributes &nbsp;|&nbsp; **Level 6**

**Business scenario:** Marketing wants the average daily order count for holidays, other weekdays and weekends in 2024.

**Task:** Using `dim_date`, return a day type ('Holiday', 'Weekend', 'Weekday') with the number of days and average orders per day in 2024 (include days with zero orders). A holiday takes priority over weekend.

**Tables:** `dim_date`, `orders`

💡 **Hint (Question 157):** Spine of all 2024 dates, left join daily counts, classify each day with CASE, then average.

In [ ]:
%sql
-- Question 157: write your solution here


### Question 158: Time-of-day segments by payment method
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Hour bucketing &nbsp;|&nbsp; **Level 6**

**Business scenario:** Payments wants to see when each payment method is used: Night (00-05), Morning (06-11), Afternoon (12-17), Evening (18-23).

**Task:** Return `payment_method` (lowercased and trimmed) and the percentage of its orders in each of the four segments (1 decimal each).

**Tables:** `orders`

💡 **Hint (Question 158):** Bucket the hour with CASE, count by method and segment, and compute percentages within each method.

In [ ]:
%sql
-- Question 158: write your solution here


### Question 159: Month-end ordering rush
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** last_day / day-of-month logic &nbsp;|&nbsp; **Level 6**

**Business scenario:** Finance suspects customers rush to order in the last days of the month (salary-day effect).

**Task:** For each month of 2025 return `month_start`, the number of valid orders, how many were placed in the last 3 calendar days of that month, and that share as a percentage (1 decimal).

**Tables:** `orders`

💡 **Hint (Question 159):** last_day(order_date) gives the final day of the month; compare it with the order's date (a date difference of 0 to 2 days) inside a conditional count.

In [ ]:
%sql
-- Question 159: write your solution here


### Question 160: Order-to-payment latency percentiles
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Percentiles on durations &nbsp;|&nbsp; **Level 6**

**Business scenario:** The payments team wants latency distributions between ordering and the first successful payment, by payment method.

**Task:** For orders that have a successful payment (any casing), compute minutes between `order_date` and the earliest success payment. Return `payment_method` of the order (lowercased and trimmed), orders, p50 and p90 minutes (1 decimal). Exclude Cash on Delivery.

**Tables:** `orders`, `payments`

💡 **Hint (Question 160):** Pre-aggregate the earliest success per order, then join to orders and use percentile_approx.

In [ ]:
%sql
-- Question 160: write your solution here


<a id="level-7"></a>
# Level 7: Advanced Analytical SQL
**Questions 161–190** · 30 questions

*Topics:* CLV, retention, cohorts, growth, market basket, funnels, Pareto, segmentation, percentile analysis

[↑ Back to table of contents](#toc)

### Question 161: Customer lifetime value (simple)
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Customer lifetime value &nbsp;|&nbsp; **Level 7**

**Business scenario:** Marketing wants each customer's lifetime value to a very first approximation.

**Task:** For valid orders only, return `customer_id`, number of orders, `lifetime_value` (SUM of `total_amount`), `avg_order_value`, `first_order_date` and `last_order_date`. Show the top 20 by lifetime value.

**Tables:** `orders`

💡 **Hint (Question 161):** A single GROUP BY customer with several aggregates is enough; no joins needed.

In [ ]:
%sql
-- Question 161: write your solution here


### Question 162: Repeat purchase rate
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Repeat purchase rate &nbsp;|&nbsp; **Level 7**

**Business scenario:** The CEO asks: what share of buying customers came back for a second purchase?

**Task:** Among customers with at least one valid order, return the number of buyers, the number with 2 or more valid orders and the `repeat_rate_pct` (1 decimal).

**Tables:** `orders`

💡 **Hint (Question 162):** Aggregate to one row per customer first, then aggregate again with conditional counts.

In [ ]:
%sql
-- Question 162: write your solution here


### Question 163: New versus returning customers per month
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** New vs returning &nbsp;|&nbsp; **Level 7**

**Business scenario:** Growth wants each month's order mix between customers placing their first valid order and returning customers.

**Task:** For each month of 2025, return `month_start`, `orders_from_new_customers` (the order is the customer's first valid order ever), `orders_from_returning_customers` and the percentage from returning customers.

**Tables:** `orders`

💡 **Hint (Question 163):** Flag each valid order as first or not using ROW_NUMBER or a first-order-date CTE, then aggregate by month.

In [ ]:
%sql
-- Question 163: write your solution here


### Question 164: Year-over-year revenue growth
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Revenue growth &nbsp;|&nbsp; **Level 7**

**Business scenario:** Investors want annual revenue and growth.

**Task:** Return `order_year`, valid-order revenue, previous year's revenue and `yoy_growth_pct` (1 decimal) for 2022-2025.

**Tables:** `orders`

💡 **Hint (Question 164):** Annual aggregation followed by LAG over the year.

In [ ]:
%sql
-- Question 164: write your solution here


### Question 165: Category contribution at the top level
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Category contribution &nbsp;|&nbsp; **Level 7**

**Business scenario:** The board wants revenue by top-level category (Electronics, Fashion, ...). Leaf categories hang under mid-level parents.

**Task:** Map each product's leaf category up to its top-level (root) category, then return `root_category`, item revenue (valid orders) and `pct_of_total` (1 decimal), largest first.

**Tables:** `order_items`, `orders`, `products`, `categories`

💡 **Hint (Question 165):** Leaf categories can be 1, 2 or 3 levels below a root. Build a mapping leaf -> root first (self-joins or recursion), then aggregate.

In [ ]:
%sql
-- Question 165: write your solution here


### Question 166: Product profitability scorecard
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Product performance &nbsp;|&nbsp; **Level 7**

**Business scenario:** Merchandising wants units, revenue, cost and gross profit for each product.

**Task:** For valid orders, return `product_id`, `units_sold`, `revenue` (SUM line_total), `cost` (SUM quantity * `unit_cost` from products), `gross_profit` and `margin_pct` (1 decimal). Show the top 20 by `gross_profit`.

**Tables:** `order_items`, `orders`, `products`

💡 **Hint (Question 166):** Gross profit = revenue - cost. Cost uses the product's unit_cost multiplied by quantity.

In [ ]:
%sql
-- Question 166: write your solution here


### Question 167: Top 5 products per top-level category by gross profit
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Top N per group &nbsp;|&nbsp; **Level 7**

**Business scenario:** Each category head wants their five most profitable products.

**Task:** Rank products by gross profit (valid orders; cost from `unit_cost`) within each top-level (root) category and return the top 5 per root category with the profit and rank.

**Tables:** `order_items`, `orders`, `products`, `categories`

💡 **Hint (Question 167):** Compute profit per product, attach the root category, then rank with a window function.

In [ ]:
%sql
-- Question 167: write your solution here


### Question 168: How many products make up 80% of revenue?
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Pareto analysis &nbsp;|&nbsp; **Level 7**

**Business scenario:** The assortment team wants to know how concentrated sales are.

**Task:** Rank products by valid item revenue, compute cumulative revenue share and return the number of products needed to reach 80% of total revenue together with that number as a percentage of all products that sold.

**Tables:** `order_items`, `orders`

💡 **Hint (Question 168):** Running SUM over the ranked products divided by the grand total; count rows up to the first row at or above 0.8.

In [ ]:
%sql
-- Question 168: write your solution here


### Question 169: Customer value segmentation
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Revenue segmentation &nbsp;|&nbsp; **Level 7**

**Business scenario:** CRM wants three value tiers based on lifetime valid revenue: High (>= 100,000), Medium (20,000 to 99,999.99), Low (below 20,000).

**Task:** Return the tier, number of customers, their combined revenue and each tier's share of total revenue (1 decimal). Include only customers with at least one valid order.

**Tables:** `orders`

💡 **Hint (Question 169):** Aggregate per customer, label with CASE, then aggregate per tier.

In [ ]:
%sql
-- Question 169: write your solution here


### Question 170: Payment method share of orders
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Percentage of total &nbsp;|&nbsp; **Level 7**

**Business scenario:** The payments lead wants the share of valid orders by payment method after tidying the labels.

**Task:** Standardise `payment_method` with lower() and trim(), then return the method, number of valid orders and `pct_of_orders` (2 decimals), largest first.

**Tables:** `orders`

💡 **Hint (Question 170):** A window SUM() OVER () over the grouped result gives the grand total for the percentage.

In [ ]:
%sql
-- Question 170: write your solution here


### Question 171: Likely churned customers
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Churn indicators &nbsp;|&nbsp; **Level 7**

**Business scenario:** Retention defines a likely-churned customer as one with at least 2 valid orders who has not placed a valid order in the 180 days up to 2025-12-31 (last order before 2025-07-04).

**Task:** Return the number of such customers and, for the 20 with the highest lifetime value, `customer_id`, `orders`, `lifetime_value`, `last_order_date` and `days_since_last_order`.

**Tables:** `orders`

💡 **Hint (Question 171):** Aggregate per customer, then filter on count and last order date. Days since is measured to 2025-12-31.

In [ ]:
%sql
-- Question 171: write your solution here


### Question 172: CRM flag versus real behaviour
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Identifying inactive customers &nbsp;|&nbsp; **Level 7**

**Business scenario:** CRM says `is_active` but the data may disagree. Behavioural activity = at least one order OR one activity event in 2025.

**Task:** Return a 2x2 table: `is_active` (CRM flag) versus behaviourally active in 2025 (TRUE/FALSE), with the number of customers in each cell.

**Tables:** `customers`, `orders`, `customer_activity`

💡 **Hint (Question 172):** Build a per-customer behavioural flag with EXISTS or LEFT JOIN to pre-aggregated 2025 data, then group by both flags.

In [ ]:
%sql
-- Question 172: write your solution here


### Question 173: Signup cohorts and first purchase
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Cohort basics &nbsp;|&nbsp; **Level 7**

**Business scenario:** Growth wants, per monthly signup cohort in 2024, the cohort size and how many customers ever placed a valid order within 30 days of signup.

**Task:** Return `cohort_month`, `cohort_size`, `buyers_within_30d` and `conversion_pct` (1 decimal).

**Tables:** `customers`, `orders`

💡 **Hint (Question 173):** Join customers to their earliest valid order date; compare with signup_date + 30.

In [ ]:
%sql
-- Question 173: write your solution here


### Question 174: Cohort order matrix for the first six months
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Cohort analysis &nbsp;|&nbsp; **Level 7**

**Business scenario:** Analytics wants orders per cohort by months since signup.

**Task:** For customers who signed up in 2024, return `cohort_month`, `months_since_signup` (0-6) and the number of valid orders in that month offset.

**Tables:** `customers`, `orders`

💡 **Hint (Question 174):** months_between or a month-index difference between order month and signup month; filter to 0-6.

In [ ]:
%sql
-- Question 174: write your solution here


### Question 175: Average order value by customer tenure
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Tenure analysis &nbsp;|&nbsp; **Level 7**

**Business scenario:** Finance wants to know if longer-tenured customers spend more per order.

**Task:** Bucket each valid order by the customer's tenure at order time: '0-3 months', '4-12 months', '13-24 months', '25+ months'. Return the bucket, number of orders and average order value (2 decimals).

**Tables:** `orders`, `customers`

💡 **Hint (Question 175):** Tenure at order time = order date minus signup_date; CASE the bucket on months_between.

In [ ]:
%sql
-- Question 175: write your solution here


### Question 176: Seasonality index
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Seasonality &nbsp;|&nbsp; **Level 7**

**Business scenario:** Planning wants to know which months run above average.

**Task:** For each full year 2022-2025, compute monthly valid revenue and a seasonality index = month revenue / that year's average monthly revenue. Return year, month, revenue and `seasonality_index` (2 decimals).

**Tables:** `orders`

💡 **Hint (Question 176):** AVG(revenue) OVER (PARTITION BY year) on the monthly series gives the yearly average.

In [ ]:
%sql
-- Question 176: write your solution here


### Question 177: Monthly retention by signup cohort
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Cohort retention &nbsp;|&nbsp; **Level 7**

**Business scenario:** Growth wants a classic retention matrix: of customers who signed up in a month, how many placed a valid order n months later?

**Task:** For 2024 signup cohorts, return `cohort_month`, `month_index` (0-6 months after the signup month), `active_customers` (distinct customers with a valid order in that month) and `retention_pct` of the cohort size (1 decimal).

**Tables:** `customers`, `orders`

💡 **Hint (Question 177):** Calculate cohort size per cohort month, active distinct customers per cohort and month index, then divide.

In [ ]:
%sql
-- Question 177: write your solution here


### Question 178: Products often bought together
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Market basket analysis &nbsp;|&nbsp; **Level 7**

**Business scenario:** Merchandising wants the product pairs most frequently found in the same order.

**Task:** Find the top 15 pairs of different products (`product_a` < `product_b`) that appear together in the same valid order, with the number of orders and each product's name. Count each order once per pair.

**Tables:** `order_items`, `orders`, `products`

💡 **Hint (Question 178):** Self-join order_items on order_id with a.product_id < b.product_id, count distinct orders.

In [ ]:
%sql
-- Question 178: write your solution here


### Question 179: Category affinity
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Market basket (categories) &nbsp;|&nbsp; **Level 7**

**Business scenario:** Cross-category promotions need the category pairs that co-occur in the same order.

**Task:** Return the top 10 pairs of different leaf categories bought in the same valid order with the number of orders and the lift-like ratio: pair orders divided by the smaller of the two categories' order counts (2 decimals).

**Tables:** `order_items`, `orders`, `products`, `categories`

💡 **Hint (Question 179):** Reduce each order to its distinct categories first, then self-join; compute single-category counts in another CTE.

In [ ]:
%sql
-- Question 179: write your solution here


### Question 180: Conversion funnel by device
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Conversion analysis &nbsp;|&nbsp; **Level 7**

**Business scenario:** Product analytics wants stage-to-stage conversion by device in 2025.

**Task:** At session level (2025 events), flag whether a session reached 'product_view', 'add_to_cart', 'checkout_start' and 'purchase'. Return `device` (lowercased/trimmed; NULL = 'unknown'), sessions, and conversion percentages for view->cart, cart->checkout and checkout->purchase (1 decimal).

**Tables:** `customer_activity`

💡 **Hint (Question 180):** Session-level conditional flags with MAX(CASE ...), then ratios of sums by device.

In [ ]:
%sql
-- Question 180: write your solution here


### Question 181: Cart abandoners without an order that month
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Activity vs sales &nbsp;|&nbsp; **Level 7**

**Business scenario:** Retention wants logged-in customers who added to cart at least 3 times in a month but placed no order in that same month.

**Task:** Return `customer_id`, `activity_month` and `add_to_cart_events` for 2025 months, for customers with at least 3 add_to_cart events and no valid order that month. Show the top 25 by events.

**Tables:** `customer_activity`, `orders`

💡 **Hint (Question 181):** Group activity by customer and month, then anti-join to monthly order existence.

In [ ]:
%sql
-- Question 181: write your solution here


### Question 182: Top 3 products each month of 2025
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Top N per group &nbsp;|&nbsp; **Level 7**

**Business scenario:** Marketing wants a monthly bestseller wall.

**Task:** For each month of 2025, return the top 3 products by item revenue (valid orders) with `month_start`, `product_id`, `product_name`, `revenue`, `rank`. Ties are broken by `product_id`.

**Tables:** `order_items`, `orders`, `products`

💡 **Hint (Question 182):** Aggregate by month and product, then ROW_NUMBER partitioned by month.

In [ ]:
%sql
-- Question 182: write your solution here


### Question 183: Top 5% of customers and their revenue share
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Percentile analysis &nbsp;|&nbsp; **Level 7**

**Business scenario:** Finance wants to know how much of revenue comes from the top five percent of customers.

**Task:** Rank buyers by lifetime valid revenue, flag the top 5% (using NTILE(20) = 1 or PERCENT_RANK), and return the number of customers in that group, their revenue and their share of total revenue (1 decimal).

**Tables:** `orders`

💡 **Hint (Question 183):** Percentile bucketing after aggregation; compare the group's sum with the overall sum.

In [ ]:
%sql
-- Question 183: write your solution here


### Question 184: CRM segment versus actual value tier
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Segmentation cross-check &nbsp;|&nbsp; **Level 7**

**Business scenario:** Loyalty wants to know whether CRM segments (Bronze/Silver/Gold/Platinum) match actual spend tiers.

**Task:** Standardise `customer_segment` (trim, initcap; NULL = 'Unknown'). Build spend quartiles from lifetime valid revenue (Q1 = top 25%). Return a cross-tab: segment x quartile with customer counts.

**Tables:** `customers`, `orders`

💡 **Hint (Question 184):** NTILE on per-customer spend, join to customers, then pivot with conditional aggregation.

In [ ]:
%sql
-- Question 184: write your solution here


### Question 185: Discount depth versus returns
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Discount effectiveness &nbsp;|&nbsp; **Level 7**

**Business scenario:** Pricing wants to see whether deeper discounts lead to more returns.

**Task:** Bucket order lines by discount percentage (0%, 1-9%, 10-19%, 20%+) of the gross line value; return the bucket, lines, units, average line total and the unit return rate (returned units / units, 2 decimals). Use lines of 'Delivered' or 'Returned' orders.

**Tables:** `order_items`, `orders`, `returns`

💡 **Hint (Question 185):** Compute discount pct per line, bucket with CASE, LEFT JOIN returns and aggregate.

In [ ]:
%sql
-- Question 185: write your solution here


### Question 186: Net revenue after refunds by month
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Net revenue &nbsp;|&nbsp; **Level 7**

**Business scenario:** Finance wants item revenue minus refunds that were paid out, by month of the original order.

**Task:** By month of `order_date`, return item revenue of valid orders, refunded amount (`return_status` = 'Refunded') on those orders' lines, `net_revenue` and `refund_rate_pct` (2 decimals), for 2025.

**Tables:** `orders`, `order_items`, `returns`

💡 **Hint (Question 186):** Aggregate returns to order_item level (or order level) before joining to avoid double counting.

In [ ]:
%sql
-- Question 186: write your solution here


### Question 187: Days from first to second order distribution
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Repeat purchase interval &nbsp;|&nbsp; **Level 7**

**Business scenario:** CRM wants the typical time between a customer's first and second valid order.

**Task:** For customers with at least 2 valid orders, bucket the days between order 1 and order 2 into 0-7, 8-30, 31-90, 91-180, 181+ and return counts and percentages (1 decimal).

**Tables:** `orders`

💡 **Hint (Question 187):** ROW_NUMBER to isolate orders 1 and 2, pivot into one row, compute the difference and bucket it.

In [ ]:
%sql
-- Question 187: write your solution here


### Question 188: Reactivated customers per year
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Win-back analysis &nbsp;|&nbsp; **Level 7**

**Business scenario:** Retention wants to know how many customers returned after being silent for 180+ days, per year.

**Task:** Using valid orders, count the distinct customers who placed an order at least 180 days after their previous valid order, grouped by the year of the returning order.

**Tables:** `orders`

💡 **Hint (Question 188):** LAG within customer to get the previous order date, filter on the gap, then count distinct customers by year.

In [ ]:
%sql
-- Question 188: write your solution here


### Question 189: Revenue concentration by customer decile
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Ranking and percentile analysis &nbsp;|&nbsp; **Level 7**

**Business scenario:** Finance wants a Lorenz-style view: how much revenue does each customer decile generate?

**Task:** Rank buyers by lifetime valid revenue, split them into 10 equal groups with NTILE(10) (decile 1 = highest spenders) and return `decile`, customers, revenue, `pct_of_total_revenue` and `cumulative_pct` (1 decimal).

**Tables:** `orders`

💡 **Hint (Question 189):** Per-customer totals, NTILE(10), group by decile, then a running SUM over the decile order.

In [ ]:
%sql
-- Question 189: write your solution here


### Question 190: Which cities drove 2025 growth?
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Contribution analysis &nbsp;|&nbsp; **Level 7**

**Business scenario:** The regional director wants the cities that contributed most to revenue growth between 2024 and 2025.

**Task:** Return customer `city` (trimmed and title-cased), 2024 revenue, 2025 revenue, absolute change and that city's share of the total change. Show the top 10 contributors.

**Tables:** `orders`, `customers`

💡 **Hint (Question 190):** Revenue by city and year in one conditional aggregation; share = city delta / total delta.

In [ ]:
%sql
-- Question 190: write your solution here


<a id="level-8"></a>
# Level 8: Data Cleaning and Data Quality
**Questions 191–210** · 20 questions

*Topics:* duplicates, NULLs, COALESCE / NULLIF, string functions, regex, standardisation, integrity and consistency checks

[↑ Back to table of contents](#toc)

### Question 191: Missing and malformed emails
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** NULL handling, pattern checks &nbsp;|&nbsp; **Level 8**

**Business scenario:** CRM wants to understand email quality.

**Task:** Return the number of customers with a NULL email, the number with an email that does not contain '@', and the number with a usable-looking email (contains '@' and not NULL).

**Tables:** `customers`

💡 **Hint (Question 191):** Use conditional aggregation with IS NULL and NOT LIKE '%@%'.

In [ ]:
%sql
-- Question 191: write your solution here


### Question 192: Standardise gender values
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** CASE WHEN + string functions &nbsp;|&nbsp; **Level 8**

**Business scenario:** Analytics needs a single clean gender attribute.

**Task:** Map the stored values (various spellings and case) to 'Female', 'Male', 'Non-binary' or 'Unknown'. Return the standardised value and the customer count. First inspect the distinct raw values.

**Tables:** `customers`

💡 **Hint (Question 192):** lower(trim(gender)) then CASE on the recognised variants such as 'f', 'female', 'm', 'male'.

In [ ]:
%sql
-- Question 192: write your solution here


### Question 193: Trim and case-fix customer names
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** TRIM, INITCAP &nbsp;|&nbsp; **Level 8**

**Business scenario:** The mailing vendor needs tidy names.

**Task:** For customers whose `first_name` or `last_name` has leading/trailing spaces or is all upper/lower case, return `customer_id`, the raw names and cleaned names (trimmed, first letter capitalised). Show 25 rows.

**Tables:** `customers`

💡 **Hint (Question 193):** Compare the value with its trimmed form; initcap(lower(trim(x))) gives a tidy result.

In [ ]:
%sql
-- Question 193: write your solution here


### Question 194: Standardise city names
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** String standardisation + mapping &nbsp;|&nbsp; **Level 8**

**Business scenario:** Regional reporting needs one spelling per city (e.g. Bangalore and Bengaluru are the same city).

**Task:** Clean `city` (trim, title-case) and map known aliases: 'Bangalore' -> 'Bengaluru', 'Bombay' -> 'Mumbai', 'Gurgaon' -> 'Gurugram', 'New Delhi' -> 'Delhi', 'Calcutta' -> 'Kolkata', 'Madras' -> 'Chennai'. Return `clean_city` and customer count, largest first.

**Tables:** `customers`

💡 **Hint (Question 194):** initcap(trim(city)) first, then a CASE (or a joined VALUES mapping) for aliases.

In [ ]:
%sql
-- Question 194: write your solution here


### Question 195: Replace NULL brand and discount
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** COALESCE &nbsp;|&nbsp; **Level 8**

**Business scenario:** Reporting must not show blanks for missing brands, and discount sums must treat NULL as zero.

**Task:** Return (a) product counts by `brand` where NULL is shown as 'Unknown' and (b) the total of `discount_amount` on orders with NULL treated as 0 versus the plain SUM.

**Tables:** `products`, `orders`

💡 **Hint (Question 195):** COALESCE(col, default). SUM ignores NULLs but AVG results change depending on how you treat them.

In [ ]:
%sql
-- Question 195: write your solution here


### Question 196: Safe division with NULLIF
**Difficulty:** Intermediate &nbsp;|&nbsp; **Topic:** NULLIF &nbsp;|&nbsp; **Level 8**

**Business scenario:** Product return rates blow up when a product sold zero units.

**Task:** For every product with at least one inventory row, return `product_id`, units sold (0 if none), units returned (0 if none) and `return_rate_pct` that is NULL instead of an error when there are no sales. Use NULLIF in the denominator. Show 25 products ordered by units sold descending.

**Tables:** `products`, `order_items`, `returns`, `inventory`

💡 **Hint (Question 196):** x / NULLIF(y, 0) returns NULL when y is 0. Aggregate sales and returns per product before joining.

In [ ]:
%sql
-- Question 196: write your solution here


### Question 197: People registered more than once
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Duplicate identification &nbsp;|&nbsp; **Level 8**

**Business scenario:** Data stewardship wants every group of customer records that share the same cleaned email.

**Task:** Return the cleaned email (lower/trim), the number of records, and the list of `customer_id`s (collect_list or concat) for emails that occur more than once. Show the 20 largest groups.

**Tables:** `customers`

💡 **Hint (Question 197):** GROUP BY lower(trim(email)) HAVING COUNT(*) > 1; collect_list aggregates the ids.

In [ ]:
%sql
-- Question 197: write your solution here


### Question 198: Golden customer records
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Deduplication &nbsp;|&nbsp; **Level 8**

**Business scenario:** Build the deduplicated customer list: one record per cleaned email, keeping the earliest signup. Customers without an email are all kept.

**Task:** Return the number of rows before and after deduplication and the number of removed duplicates. Also return 10 sample removed `customer_id`s with the id of the surviving record.

**Tables:** `customers`

💡 **Hint (Question 198):** ROW_NUMBER over the cleaned email (NULL emails need a separate rule), keep rn = 1.

In [ ]:
%sql
-- Question 198: write your solution here


### Question 199: Duplicate order lines
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Duplicate detection + impact &nbsp;|&nbsp; **Level 8**

**Business scenario:** The data team suspects some order lines were loaded twice.

**Task:** Find (order_id, product_id, quantity, unit_price, discount) combinations that appear more than once. Return the number of duplicate groups, the number of extra rows and the revenue inflation (SUM of `line_total` of the extra rows).

**Tables:** `order_items`

💡 **Hint (Question 199):** Group on all business-key columns, count extras as COUNT(*) - 1 per group, then sum.

In [ ]:
%sql
-- Question 199: write your solution here


### Question 200: Email format validation
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Regular expressions &nbsp;|&nbsp; **Level 8**

**Business scenario:** The marketing automation tool rejects malformed emails.

**Task:** Using a regular expression (RLIKE / regexp_like), classify non-NULL emails as 'valid' (text@domain.tld pattern after trimming and lower-casing) or 'invalid'. Return the counts and 15 sample invalid emails.

**Tables:** `customers`

💡 **Hint (Question 200):** A simple pattern like ^[a-z0-9._%+-]+@[a-z0-9.-]+\\.[a-z]{2,}$ is enough; mind escaping in Spark SQL strings.

In [ ]:
%sql
-- Question 200: write your solution here


### Question 201: Extract the email domain
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** regexp_extract &nbsp;|&nbsp; **Level 8**

**Business scenario:** Partnerships wants the top email providers.

**Task:** Extract the domain from each valid-looking email (after trim + lower) and return the 10 most common domains with counts and percentage of customers with an email.

**Tables:** `customers`

💡 **Hint (Question 201):** Pull out everything after the @ sign with regexp_extract or split_part, then lower-case it before grouping and counting.

In [ ]:
%sql
-- Question 201: write your solution here


### Question 202: Implausible dates of birth
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Invalid value detection &nbsp;|&nbsp; **Level 8**

**Business scenario:** Compliance needs to find unrealistic dates of birth.

**Task:** Return the number of customers whose `date_of_birth` is NULL, in the future relative to 2025-12-31, before 1930-01-01, or makes the customer younger than 16 at signup. Show each category and 5 sample rows each.

**Tables:** `customers`

💡 **Hint (Question 202):** Several CASE branches (or UNION ALL of rules) produce an issue label per rule.

In [ ]:
%sql
-- Question 202: write your solution here


### Question 203: Orders missing items, payments or products
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Missing relationships &nbsp;|&nbsp; **Level 8**

**Business scenario:** Data quality wants a summary of unexpected missing child records.

**Task:** Return the counts of: orders with no order lines, 'Delivered' or 'Returned' orders with no payment row, and products that have no inventory row.

**Tables:** `orders`, `order_items`, `payments`, `products`, `inventory`

💡 **Hint (Question 203):** Three anti-joins (NOT EXISTS) combined into one result via UNION ALL or scalar subqueries.

In [ ]:
%sql
-- Question 203: write your solution here


### Question 204: Order header versus line mismatch
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Consistency checks &nbsp;|&nbsp; **Level 8**

**Business scenario:** Finance needs to know which order totals do not equal items + tax + shipping.

**Task:** Return `order_id`, header `total_amount`, computed total (SUM(`line_total`) + `tax_amount` + `shipping_amount`) and the difference for orders where the absolute difference exceeds 1 INR. Return also the number of mismatched orders. Show 20 rows ordered by absolute difference.

**Tables:** `orders`, `order_items`

💡 **Hint (Question 204):** Pre-aggregate the lines per order before joining; also consider orders with no lines.

In [ ]:
%sql
-- Question 204: write your solution here


### Question 205: Short and over payments
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Reconciliation checks &nbsp;|&nbsp; **Level 8**

**Business scenario:** Treasury wants successful payments that differ from the order total.

**Task:** For orders with exactly one successful payment (status 'Success' in any casing), return the number of orders where the payment is lower than `total_amount` by more than 1 INR, higher by more than 1 INR, or equal within 1 INR.

**Tables:** `orders`, `payments`

💡 **Hint (Question 205):** Count successful payments per order, keep those with 1, compare amounts with CASE.

In [ ]:
%sql
-- Question 205: write your solution here


### Question 206: Return quantity and refund anomalies
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Business rule validation &nbsp;|&nbsp; **Level 8**

**Business scenario:** Quality wants returns that break basic business rules.

**Task:** Return counts of returns where (a) `returned_quantity` exceeds the line's purchased `quantity`, (b) `refund_amount` exceeds the line's `line_total`, and (c) `return_date` is earlier than the order date.

**Tables:** `returns`, `order_items`, `orders`

💡 **Hint (Question 206):** Join returns to order_items and orders and use three conditional counts.

In [ ]:
%sql
-- Question 206: write your solution here


### Question 207: Sales outside the product's life
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Temporal consistency &nbsp;|&nbsp; **Level 8**

**Business scenario:** Catalogue integrity wants order lines for products sold before their launch or after their discontinuation.

**Task:** Return the number of lines and revenue sold before `launch_date` and after `discontinued_date`, plus 10 sample rows with product name and order date.

**Tables:** `order_items`, `orders`, `products`

💡 **Hint (Question 207):** Compare the date of order_date to products.launch_date and discontinued_date.

In [ ]:
%sql
-- Question 207: write your solution here


### Question 208: Repeated failed payments in a day
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Suspicious pattern detection &nbsp;|&nbsp; **Level 8**

**Business scenario:** Fraud wants customers with 5 or more failed payment attempts on the same calendar day.

**Task:** Join payments to orders to get the customer. Count failed attempts (status 'Failed' in any casing) per customer per day and return customers/days with at least 5 failures, with the number of distinct orders and distinct payment methods involved.

**Tables:** `payments`, `orders`

💡 **Hint (Question 208):** Group by customer and payment date, count failures with a conditional aggregate, and filter with HAVING.

In [ ]:
%sql
-- Question 208: write your solution here


### Question 209: Unusual prices and quantities
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Outlier detection &nbsp;|&nbsp; **Level 8**

**Business scenario:** Pricing wants order lines far away from the product's normal price or with extreme quantities.

**Task:** Return lines where `unit_price` is at least 5 times the product's current `selling_price` OR `quantity` >= 10. Include `order_item_id`, product name, quantity, unit price, selling price, ratio and line total. Show the 25 largest by line total.

**Tables:** `order_items`, `products`

💡 **Hint (Question 209):** Join to products and compute the price ratio; combine both rules in one WHERE with OR.

In [ ]:
%sql
-- Question 209: write your solution here


### Question 210: Inventory data integrity report
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Multi-rule data quality query &nbsp;|&nbsp; **Level 8**

**Business scenario:** The inventory team wants a single issue report for stock records.

**Task:** Return a table of issue type and affected row count for: negative stock, reserved greater than on-hand, zero stock with reserved > 0, rows not updated since before 2025-12-01, and active products without any inventory row.

**Tables:** `inventory`, `products`

💡 **Hint (Question 210):** One SELECT per rule combined with UNION ALL keeps the report easy to extend.

In [ ]:
%sql
-- Question 210: write your solution here


<a id="level-9"></a>
# Level 9: Data Warehousing and Analytics Engineering SQL
**Questions 211–230** · 20 questions

*Topics:* grain, facts, dimensions, surrogate keys, SCD 1/2, incremental logic, late-arriving data, reconciliation, dbt-style tests

[↑ Back to table of contents](#toc)

### Question 211: Is the grain really one row per order line?
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Identifying table grain &nbsp;|&nbsp; **Level 9**

**Business scenario:** Before building `fact_order_items` you must prove what one row represents.

**Task:** Return `total_rows`, `distinct_order_item_id`, and `distinct_order_product_pairs` (distinct `order_id`, `product_id`) for `order_items`. Then list 10 (`order_id`, `product_id`) pairs that occur more than once, with how many rows each has.

**Tables:** `order_items`

💡 **Hint (Question 211):** Compare COUNT(*) with COUNT(DISTINCT ...) of candidate keys, then GROUP BY the candidate key HAVING COUNT(*) > 1.

In [ ]:
%sql
-- Question 211: write your solution here


### Question 212: stg_customers: a cleaned staging model
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Staging / standardisation &nbsp;|&nbsp; **Level 9**

**Business scenario:** Write the SELECT behind a dbt staging model for customers with consistent formats.

**Task:** Return one row per `customer_id` with: `email_clean` (lower, trimmed, NULL if invalid), `first_name_clean`, `last_name_clean` (trimmed, title-case), `gender_clean` (Female/Male/Non-binary/Unknown), `segment_clean` (title-case, 'Unknown' when NULL), `city_clean` (trimmed, title-case), `country`, `signup_date`, `is_active`, and `is_email_valid`. Show 25 rows; also return the total row count to prove the grain is unchanged.

**Tables:** `customers`

💡 **Hint (Question 212):** Chain trim/lower/initcap and CASE expressions inside one SELECT; keep the grain by not joining or filtering.

In [ ]:
%sql
-- Question 212: write your solution here


### Question 213: stg_payments with duplicate flag
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Staging + deduplication flag &nbsp;|&nbsp; **Level 9**

**Business scenario:** The payments staging model must standardise labels and mark duplicate callbacks rather than silently dropping them.

**Task:** Return `payment_id`, `order_id`, `payment_method_clean` (trimmed, title-case; UPI upper-case), `payment_status_clean` (title-case), `payment_amount`, `transaction_reference` and `is_duplicate_callback` (TRUE when another row with the same reference and order exists with a lower `payment_id`). Show the first 30 rows where the flag is TRUE.

**Tables:** `payments`

💡 **Hint (Question 213):** ROW_NUMBER() OVER (PARTITION BY transaction_reference, order_id ORDER BY payment_id) > 1 identifies later duplicates.

In [ ]:
%sql
-- Question 213: write your solution here


### Question 214: dim_product with the full category path
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Dimension construction &nbsp;|&nbsp; **Level 9**

**Business scenario:** Build the product dimension: one row per product with readable attributes and its category hierarchy.

**Task:** Return `product_id`, `product_name_clean` (trimmed, title-case), `brand_clean` ('Unknown' when NULL; title-case), `leaf_category`, `mid_category`, `root_category`, `unit_cost`, `selling_price`, `margin_pct` and `is_active`. Show 25 rows and confirm the total row count equals the number of products.

**Tables:** `products`, `categories`

💡 **Hint (Question 214):** Join categories up the tree (leaf -> parent -> grandparent) with LEFT JOINs and COALESCE to cope with different depths.

In [ ]:
%sql
-- Question 214: write your solution here


### Question 215: Generating surrogate keys
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Surrogate keys &nbsp;|&nbsp; **Level 9**

**Business scenario:** A customer dimension needs a warehouse key independent of the source id.

**Task:** Build two surrogate key candidates for `customers`: `customer_key_seq` using ROW_NUMBER ordered by `customer_id`, and `customer_key_hash` using a hash/md5 of `customer_id` (any deterministic function). Return both keys for 10 customers and prove uniqueness by comparing COUNT(*) with COUNT(DISTINCT key) for each.

**Tables:** `customers`

💡 **Hint (Question 215):** ROW_NUMBER is stable only for a fixed ordering; hashes are stable across loads. Uniqueness tests compare total and distinct counts.

In [ ]:
%sql
-- Question 215: write your solution here


### Question 216: fact_order_items with dimension keys
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Fact table construction &nbsp;|&nbsp; **Level 9**

**Business scenario:** Build the order-line fact: grain = one row per order line, with keys to the date, customer and product dimensions.

**Task:** Return `order_item_id`, `order_date_key` (yyyymmdd integer matching `dim_date.date_key`), `customer_id`, `product_id`, `quantity`, `gross_amount` (quantity * unit_price), `discount_amount`, `net_amount` (`line_total`), `cost_amount` (quantity * unit_cost) and `gross_profit`. Show 20 rows and verify the row count equals `order_items`.

**Tables:** `order_items`, `orders`, `products`, `dim_date`

💡 **Hint (Question 216):** Join order_items to orders (date, customer), products (cost) and optionally dim_date to validate the key. All joins must be many-to-one to preserve the grain.

In [ ]:
%sql
-- Question 216: write your solution here


### Question 217: fact_orders without fan-out
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Fact table at order grain &nbsp;|&nbsp; **Level 9**

**Business scenario:** Build an order-grain fact that carries item and payment aggregates safely.

**Task:** Return one row per order: `order_id`, `customer_id`, `order_status`, `order_total`, `line_count`, `units`, `items_amount`, `paid_amount` (successful payments), `refunded_payment_amount` (payments with status 'Refunded') and `refund_amount_returns` (returns refunded). Missing children must show 0. Show 20 rows and confirm the row count equals `orders`.

**Tables:** `orders`, `order_items`, `payments`, `returns`

💡 **Hint (Question 217):** Pre-aggregate each child table to order level in CTEs, then LEFT JOIN each to orders and COALESCE to 0.

In [ ]:
%sql
-- Question 217: write your solution here


### Question 218: SCD Type 1: what would be overwritten?
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Slowly changing dimensions (Type 1) &nbsp;|&nbsp; **Level 9**

**Business scenario:** A Type 1 refresh overwrites customer attributes with the latest source values from the change feed.

**Task:** For each customer take the most recent `customer_changes` row (by `change_timestamp`, then `change_id`). Return how many customers have a `city` or `customer_segment` in `customers` that differs from that latest record after trimming/title-casing both, and show 15 examples with old and new values.

**Tables:** `customers`, `customer_changes`

💡 **Hint (Question 218):** Rank change rows per customer, keep rank 1, join to customers and compare cleaned values.

In [ ]:
%sql
-- Question 218: write your solution here


### Question 219: SCD Type 2: build the history table
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Slowly changing dimensions (Type 2) &nbsp;|&nbsp; **Level 9**

**Business scenario:** Turn the change feed into a Type 2 customer dimension with validity ranges.

**Task:** Remove exact duplicate feed rows, then return `customer_id`, `city`, `state`, `customer_segment`, `is_active`, `valid_from` (change_timestamp), `valid_to` (the next version's `valid_from`, or '9999-12-31' for the current one) and `is_current`. Show all versions for customers 1, 2 and 3 plus the total number of rows and current rows.

**Tables:** `customer_changes`

💡 **Hint (Question 219):** LEAD(change_timestamp) OVER (PARTITION BY customer_id ORDER BY change_timestamp) provides valid_to. Dedupe with ROW_NUMBER or DISTINCT first.

In [ ]:
%sql
-- Question 219: write your solution here


### Question 220: Point-in-time join to a Type 2 dimension
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** SCD2 joins &nbsp;|&nbsp; **Level 9**

**Business scenario:** Revenue should be attributed to the city the customer lived in when the order was placed, not today's city.

**Task:** Using the Type 2 history (from `customer_changes`) join each valid 2025 order to the customer version that was valid at `order_date`. Return revenue by historical city (title-cased) and the number of orders whose historical city differs from the customer's current city in `customers` (after cleaning).

**Tables:** `orders`, `customer_changes`, `customers`

💡 **Hint (Question 220):** Join on customer_id and order_date >= valid_from AND order_date < valid_to.

In [ ]:
%sql
-- Question 220: write your solution here


### Question 221: Late-arriving change records
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Handling late-arriving data &nbsp;|&nbsp; **Level 9**

**Business scenario:** Some change events reach the warehouse days after they happened. Ordering by arrival time would put versions in the wrong order.

**Task:** Return (a) the number of feed rows whose `received_at` is more than 24 hours after `change_timestamp`, by month of `change_timestamp`, and (b) the number of customers whose latest version by `received_at` differs from their latest version by `change_timestamp`.

**Tables:** `customer_changes`

💡 **Hint (Question 221):** Rank rows per customer twice with different ORDER BYs and compare the winners.

In [ ]:
%sql
-- Question 221: write your solution here


### Question 222: Classify the incoming batch
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Identifying new and changed records &nbsp;|&nbsp; **Level 9**

**Business scenario:** An incremental load arrives in `orders_incoming_batch`. Before merging, classify what is in it.

**Task:** For each distinct `order_id` in the batch (take the latest row by `updated_at`, then `loaded_at`), label it 'new' (not in `orders`), 'changed' (in `orders` but `order_status` or `total_amount` differs) or 'unchanged'. Return the label counts and the number of redundant duplicate rows in the batch.

**Tables:** `orders_incoming_batch`, `orders`

💡 **Hint (Question 222):** Dedupe the batch first with ROW_NUMBER, then LEFT JOIN to orders and compare columns.

In [ ]:
%sql
-- Question 222: write your solution here


### Question 223: Which incremental filter misses late data?
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Incremental strategies and watermarks &nbsp;|&nbsp; **Level 9**

**Business scenario:** Two engineers disagree: filter new data by `order_date` greater than the target's max order date, or by `updated_at` greater than the target's max `updated_at`.

**Task:** Compute both watermarks from `orders`, then return how many distinct batch orders each strategy would select and how many batch orders the `order_date` strategy would miss that the `updated_at` strategy captures.

**Tables:** `orders_incoming_batch`, `orders`

💡 **Hint (Question 223):** Scalar subqueries for the two MAX() values, then conditional counts on distinct order_ids in the batch.

In [ ]:
%sql
-- Question 223: write your solution here


### Question 224: What is the unique key of the batch?
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Incremental keys and duplicates &nbsp;|&nbsp; **Level 9**

**Business scenario:** A dbt incremental model needs a unique_key. Test whether `order_id` is safe in the batch.

**Task:** Return total batch rows, distinct `order_id`s, how many order_ids appear more than once, how many of those repeats are fully identical rows and how many differ in at least one column (e.g. `order_status` or `updated_at`).

**Tables:** `orders_incoming_batch`

💡 **Hint (Question 224):** GROUP BY order_id and compare COUNT(*) with COUNT(DISTINCT <columns>) to see whether the repeats are identical.

In [ ]:
%sql
-- Question 224: write your solution here


### Question 225: Simulate the merge
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Incremental merge logic &nbsp;|&nbsp; **Level 9**

**Business scenario:** Produce the state of `orders` after applying the batch (upsert on `order_id`, latest version wins).

**Task:** Combine target rows that are not in the batch with the latest batch row per `order_id`. Return the resulting row count, the number of inserts, the number of updates and the count of orders by `order_status` after the merge.

**Tables:** `orders`, `orders_incoming_batch`

💡 **Hint (Question 225):** UNION ALL of (target anti-joined to batch) and (deduped batch), then aggregate. Count inserts/updates by comparing to the target.

In [ ]:
%sql
-- Question 225: write your solution here


### Question 226: Late-arriving dimension rows
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Late-arriving dimensions &nbsp;|&nbsp; **Level 9**

**Business scenario:** Some batch orders reference customers who are not in `customers` yet. A fact table must not drop them.

**Task:** For deduplicated batch orders, return how many have no matching customer, their total `total_amount`, and 10 sample rows with a `customer_key` fallback of -1 and `customer_name` 'Unknown (late arriving)' for those, while matched customers keep their id and real name.

**Tables:** `orders_incoming_batch`, `customers`

💡 **Hint (Question 226):** LEFT JOIN to customers and COALESCE the missing keys/names so the fact row survives.

In [ ]:
%sql
-- Question 226: write your solution here


### Question 227: Reconcile monthly totals: header versus lines
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Data reconciliation &nbsp;|&nbsp; **Level 9**

**Business scenario:** Finance wants a monthly control report comparing the order header total with the sum of its lines + tax + shipping.

**Task:** By month of `order_date` return orders, `header_total` (SUM of `total_amount`), `lines_total` (SUM of line totals + `tax_amount` + `shipping_amount` computed per order), `difference` and `orders_with_difference` (more than 1 INR). Include orders that have no lines. Show the 12 months with the largest absolute difference.

**Tables:** `orders`, `order_items`

💡 **Hint (Question 227):** Aggregate lines to order level first, LEFT JOIN from orders, then aggregate by month.

In [ ]:
%sql
-- Question 227: write your solution here


### Question 228: A reusable label-mapping table
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Reusable transformation logic &nbsp;|&nbsp; **Level 9**

**Business scenario:** Instead of repeating CASE statements, analytics engineering keeps a mapping (like a dbt seed) and joins to it.

**Task:** Create the mapping inline with VALUES (raw lower-trimmed label -> clean label) for payment methods, apply it to both `orders.payment_method` and `payments.payment_method`, and return for each table the number of rows with an unmapped label (should be 0) and the clean label counts.

**Tables:** `orders`, `payments`

💡 **Hint (Question 228):** Inspect distinct lower(trim()) labels first. A VALUES list inside a CTE works like a seed table.

In [ ]:
%sql
-- Question 228: write your solution here


### Question 229: Daily category sales mart
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Reporting-ready dataset &nbsp;|&nbsp; **Level 9**

**Business scenario:** BI needs a zero-filled daily sales table by leaf category with month-to-date totals.

**Task:** For December 2025, build one row per calendar day x leaf category in categories 41-44 (Smartphones, Cases & Covers, Chargers & Cables, Power Banks) with `units`, `revenue` (valid orders) and `month_to_date_revenue`. Days without sales must still appear with 0.

**Tables:** `dim_date`, `categories`, `order_items`, `orders`, `products`

💡 **Hint (Question 229):** CROSS JOIN a date spine with the category list, LEFT JOIN daily facts, COALESCE, then a running SUM per category.

In [ ]:
%sql
-- Question 229: write your solution here


### Question 230: dbt-style data tests in SQL
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Data tests &nbsp;|&nbsp; **Level 9**

**Business scenario:** Write the equivalent of dbt tests (unique, not_null, accepted_values, relationships) as one query returning failures.

**Task:** Return `test_name` and `failing_rows` for: unique `orders.order_id`; not_null `orders.customer_id`; accepted_values `orders.order_status` (Pending, Processing, Shipped, Delivered, Cancelled, Returned, Failed); relationships `order_items.order_id` -> `orders`; and a custom test that no order has a negative `total_amount`.

**Tables:** `orders`, `order_items`

💡 **Hint (Question 230):** Each test is a SELECT counting the rows that violate the rule; combine them with UNION ALL.

In [ ]:
%sql
-- Question 230: write your solution here


<a id="level-10"></a>
# Level 10: Advanced Business Scenarios and Interview Challenges
**Questions 231–250** · 20 questions

*Topics:* multi-step, multi-concept business problems

[↑ Back to table of contents](#toc)

### Question 231: Customer lifetime value, net of refunds, with rank
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** CLV + ranking &nbsp;|&nbsp; **Level 10**

**Business scenario:** Finance wants CLV based on item revenue after refunds actually paid, ranked globally and within country.

**Task:** For each customer with a valid order, return `customer_id`, `country`, `item_revenue`, `refunded_amount` (returns with status 'Refunded'), `net_clv`, `global_rank` and `country_rank`. Show the top 20 globally. Avoid double counting.

**Tables:** `orders`, `order_items`, `returns`, `customers`

💡 **Hint (Question 231):** Pre-aggregate revenue and refunds to the customer level in separate CTEs, join them, then apply two RANK window functions.

In [ ]:
%sql
-- Question 231: write your solution here


### Question 232: Multi-category shoppers
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Distinct counts across a join chain &nbsp;|&nbsp; **Level 10**

**Business scenario:** Cross-sell wants customers who have bought from at least three different leaf categories.

**Task:** Using valid orders, return the number of customers who purchased from 3 or more distinct leaf categories, plus the top 15 customers by number of categories with their category count and revenue.

**Tables:** `orders`, `order_items`, `products`

💡 **Hint (Question 232):** COUNT(DISTINCT category_id) per customer after joining orders -> order_items -> products.

In [ ]:
%sql
-- Question 232: write your solution here


### Question 233: Second purchase date for every customer
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Nth event per group &nbsp;|&nbsp; **Level 10**

**Business scenario:** CRM wants the date of every customer's second valid order and how long it took.

**Task:** Return `customer_id`, `first_order_date`, `second_order_date` and `days_to_second_order` for customers with at least two valid orders (show 25 ordered by `customer_id`). Also return how many buyers never placed a second order.

**Tables:** `orders`

💡 **Hint (Question 233):** ROW_NUMBER per customer, then pivot rows 1 and 2 with conditional MAX or a self-join.

In [ ]:
%sql
-- Question 233: write your solution here


### Question 234: Rolling 30-day revenue
**Difficulty:** Advanced &nbsp;|&nbsp; **Topic:** Rolling window on a date spine &nbsp;|&nbsp; **Level 10**

**Business scenario:** The CFO wants a trailing 30-day revenue line for 2025.

**Task:** For every calendar day of 2025 (including days with no orders), return `calendar_date`, `daily_revenue` (valid orders) and `rolling_30d_revenue` covering the current day and the previous 29 days (days before 2025-01-01 count as available data from 2024).

**Tables:** `dim_date`, `orders`

💡 **Hint (Question 234):** Spine from dim_date, daily aggregate, and a window frame of 29 PRECEDING rows (or RANGE over a day number). Build the spine back to December 2024.

In [ ]:
%sql
-- Question 234: write your solution here


### Question 235: Three consecutive months of increasing spend
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Consecutive-period pattern &nbsp;|&nbsp; **Level 10**

**Business scenario:** CRM wants customers whose monthly spend rose in three consecutive calendar months (month2 > month1, month3 > month2, month4 > month3).

**Task:** Aggregate valid spend per customer and calendar month. Find customers with four consecutive calendar months of data where each month's spend is strictly higher than the previous. Return `customer_id`, the first month of the run and the four spend values. Show 20 rows.

**Tables:** `orders`

💡 **Hint (Question 235):** Make sure the months are really consecutive (check month index differences) and use LAG/LEAD or a self-join with month offsets.

In [ ]:
%sql
-- Question 235: write your solution here


### Question 236: Cohort cumulative revenue per customer
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Cohort LTV curve &nbsp;|&nbsp; **Level 10**

**Business scenario:** Finance wants to see how much revenue the average customer of each 2023 signup cohort has generated by 0, 3, 6 and 12 months after the signup month.

**Task:** For signup cohorts in 2023 (by signup month), return `cohort_month`, cohort size and cumulative valid revenue per customer at month offsets 0, 3, 6 and 12 (one column each).

**Tables:** `customers`, `orders`

💡 **Hint (Question 236):** Compute revenue by cohort and month offset, cumulative-sum it per cohort, then pivot the offsets of interest and divide by cohort size.

In [ ]:
%sql
-- Question 236: write your solution here


### Question 237: Products making up 80% of revenue in each top-level category
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Pareto within groups &nbsp;|&nbsp; **Level 10**

**Business scenario:** Each category head wants the vital few products that deliver 80% of that category's revenue.

**Task:** For each root category, rank products by valid item revenue, compute cumulative share within the category, and return the number of products that make up the first 80% along with the category's total product count that sold and the percentage.

**Tables:** `order_items`, `orders`, `products`, `categories`

💡 **Hint (Question 237):** Map leaf categories to root categories, rank within root, running SUM / total per root, then count rows up to the 80% line.

In [ ]:
%sql
-- Question 237: write your solution here


### Question 238: Orders placed but never paid
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Anti-join on a derived status &nbsp;|&nbsp; **Level 10**

**Business scenario:** Finance wants customers who tried to buy but never completed a payment on any order.

**Task:** Return customers who have at least one non-cancelled order and no 'Success' payment (any casing) on any of their orders. Show `customer_id`, number of orders, amount attempted (sum of `total_amount`) and the number of failed or pending payment rows. Show the top 20 by amount attempted.

**Tables:** `customers`, `orders`, `payments`

💡 **Hint (Question 238):** Aggregate per customer and use a NOT EXISTS (or conditional sum = 0) on successful payments. Cash on Delivery orders may have no payment rows at all.

In [ ]:
%sql
-- Question 238: write your solution here


### Question 239: Net revenue after discounts and returns
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Revenue waterfall &nbsp;|&nbsp; **Level 10**

**Business scenario:** Finance wants a waterfall per root category and quarter for 2025: gross value, discounts, refunds and net revenue.

**Task:** For lines of valid orders placed in 2025 return `root_category`, `quarter`, `gross_amount` (quantity * unit_price), `discount_amount`, `refunded_amount` (returns with status 'Refunded') and `net_revenue` (gross - discount - refunded). Order by category and quarter.

**Tables:** `order_items`, `orders`, `products`, `categories`, `returns`

💡 **Hint (Question 239):** Aggregate returns to order-line level before the join to avoid double counting; map each leaf category up to its root.

In [ ]:
%sql
-- Question 239: write your solution here


### Question 240: Inventory shortage versus sales velocity
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Days of cover &nbsp;|&nbsp; **Level 10**

**Business scenario:** Supply planning wants to flag active products that will sell out soon.

**Task:** Velocity = units sold in valid orders placed from 2025-10-02 to 2025-12-31 (91 days) / 91. Available stock = SUM(stock_quantity - reserved_quantity) across warehouses (treat negatives as 0 per row). Return products with velocity > 0 and `days_of_cover` < 14: `product_id`, `product_name`, `available_stock`, `daily_velocity` (2 decimals) and `days_of_cover` (1 decimal). Show the 25 most urgent.

**Tables:** `products`, `inventory`, `order_items`, `orders`

💡 **Hint (Question 240):** Two aggregates (sales and stock) per product, joined and divided. Use GREATEST(0, ...) at row level for stock.

In [ ]:
%sql
-- Question 240: write your solution here


### Question 241: Reconcile orders with payments and refunds
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Financial reconciliation &nbsp;|&nbsp; **Level 10**

**Business scenario:** Treasury needs to see, per order, whether cash received matches what was billed after refunds.

**Task:** For orders with status Delivered or Returned: `paid` = SUM of successful payments (any casing, duplicates by `transaction_reference` counted once), `refunded` = refunded return amounts, `expected_net` = `total_amount` - `refunded` and a `recon_status` ('Matched' within 1 INR, 'Short paid', 'Over paid', 'Unpaid'). Return counts and amount gaps by `recon_status`.

**Tables:** `orders`, `payments`, `returns`, `order_items`

💡 **Hint (Question 241):** De-duplicate payments per transaction reference first, aggregate refunds per order, then compare in a CASE.

In [ ]:
%sql
-- Question 241: write your solution here


### Question 242: Customer-level analytical dataset
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Feature table / mart &nbsp;|&nbsp; **Level 10**

**Business scenario:** Data science wants one row per customer with features. This is the classic customer mart.

**Task:** Return one row per `customer_id` (all customers, including those without orders) with: cleaned city and segment, `signup_date`, valid `orders_count`, `lifetime_revenue`, `avg_order_value`, `first_order_date`, `last_order_date`, `days_since_last_order` (at 2025-12-31), `returns_count`, `refunded_amount`, `distinct_categories`, `activity_events_2025`, and flags `is_repeat_buyer` and `is_never_bought`. Show 25 rows and confirm the row count equals `customers`.

**Tables:** `customers`, `orders`, `order_items`, `products`, `returns`, `customer_activity`

💡 **Hint (Question 242):** Build each feature block as its own CTE at customer grain, LEFT JOIN them all to customers, and COALESCE counts to 0.

In [ ]:
%sql
-- Question 242: write your solution here


### Question 243: Rule-based suspicious transaction score
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Anomaly detection with SQL &nbsp;|&nbsp; **Level 10**

**Business scenario:** Risk wants a simple score for orders that look suspicious.

**Task:** Score each valid-status order (any status except Cancelled) 1 point each for: customer had 3+ failed payment rows on the order's day; another order by the same customer within 5 minutes; `total_amount` above 100,000; shipping address NULL; payment reference reused on a different order. Return orders with a score of 2 or more with `order_id`, `customer_id`, the score and which rules fired. Show 25 by score descending.

**Tables:** `orders`, `payments`

💡 **Hint (Question 243):** Compute each rule in its own CTE keyed by order_id, join them with LEFT JOIN, and add the flags.

In [ ]:
%sql
-- Question 243: write your solution here


### Question 244: Monthly product performance dataset
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Product x month mart &nbsp;|&nbsp; **Level 10**

**Business scenario:** Build the monthly product performance table for 2025.

**Task:** Grain: product x month for 2025. Columns: `product_id`, `month_start`, `units`, `revenue`, `units_returned`, `return_rate_pct`, `revenue_rank_in_month` and `revenue_change_vs_prev_month` (NULL when no previous month). Only include product-months with sales. Show the 25 top product-months by revenue.

**Tables:** `order_items`, `orders`, `returns`

💡 **Hint (Question 244):** Aggregate lines (and returns) to product-month, then apply RANK partitioned by month and LAG partitioned by product.

In [ ]:
%sql
-- Question 244: write your solution here


### Question 245: RFM scoring
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Segmentation with NTILE &nbsp;|&nbsp; **Level 10**

**Business scenario:** Marketing wants RFM segments at 2025-12-31.

**Task:** For customers with a valid order compute Recency (days since last valid order), Frequency (valid orders) and Monetary (valid revenue). Score each 1-5 with NTILE (5 = best; best recency = lowest days). Label 'Champions' (R>=4 and F>=4 and M>=4), 'At risk' (R<=2 and F>=3), 'Lost' (R=1 and F<=2), else 'Others'. Return label, customers and average revenue.

**Tables:** `orders`

💡 **Hint (Question 245):** Per-customer R, F and M in one CTE; three NTILE calls; CASE for the label; aggregate by label.

In [ ]:
%sql
-- Question 245: write your solution here


### Question 246: Time to first purchase by signup quarter
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Cohort timing &nbsp;|&nbsp; **Level 10**

**Business scenario:** Growth wants to know how fast new customers make their first purchase.

**Task:** For signup quarters in 2023-2024, return `signup_quarter`, customers, `never_purchased_pct` (no valid order ever), `avg_days_to_first_order` and `median_days_to_first_order` for those who purchased.

**Tables:** `customers`, `orders`

💡 **Hint (Question 246):** LEFT JOIN each customer to the first valid order date; percentile functions work on the purchased subset.

In [ ]:
%sql
-- Question 246: write your solution here


### Question 247: Customers who switched favourite category
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Year-over-year behaviour change &nbsp;|&nbsp; **Level 10**

**Business scenario:** CRM wants customers whose favourite leaf category (by valid item revenue) in 2025 differs from 2024.

**Task:** For customers with valid orders in both years compute the top category per year (tie: lower `category_id`). Return the number of customers whose top category changed, and the 10 most common (from -> to) switches.

**Tables:** `orders`, `order_items`, `products`, `categories`

💡 **Hint (Question 247):** Rank categories per customer per year, keep rank 1, then join 2024 to 2025 on customer.

In [ ]:
%sql
-- Question 247: write your solution here


### Question 248: Does a heavy discount on the first order drive repeat purchases?
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Experiment-style analysis &nbsp;|&nbsp; **Level 10**

**Business scenario:** Pricing asks whether a deeply discounted first order leads to a second order within 90 days.

**Task:** For each customer's first valid order placed before 2025-10-01, compute its discount rate = SUM(`discount`) / SUM(`quantity` * `unit_price`) over its order lines. Compare customers with a rate >= 15% against the rest: customers, those with another valid order within 90 days, and the repeat percentage (1 decimal).

**Tables:** `orders`, `order_items`

💡 **Hint (Question 248):** Get the first order per customer, its discount rate from the lines, then check for a later valid order in a 90-day window.

In [ ]:
%sql
-- Question 248: write your solution here


### Question 249: Return hot-spots
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** Multi-step product quality analysis &nbsp;|&nbsp; **Level 10**

**Business scenario:** Quality assurance wants products with a high return rate and the dominant reason.

**Task:** For products with at least 30 units sold (all order lines), compute the returned-unit rate. For those at 15% or higher return `product_id`, `product_name`, `category_name`, units sold, return rate (1 decimal) and the most frequent non-NULL `return_reason` (tie: alphabetical).

**Tables:** `order_items`, `returns`, `products`, `categories`

💡 **Hint (Question 249):** Aggregate sales and returns per product; find the top reason per product with ROW_NUMBER over reason counts.

In [ ]:
%sql
-- Question 249: write your solution here


### Question 250: Monthly executive scorecard
**Difficulty:** Expert &nbsp;|&nbsp; **Topic:** KPI dataset &nbsp;|&nbsp; **Level 10**

**Business scenario:** Leadership wants one table of core KPIs by month for 2025.

**Task:** One row per month: `orders` (all statuses), `valid_revenue` (`total_amount` of valid orders), `aov`, `new_customers` (first valid order in the month), `returning_order_share_pct`, `cancel_rate_pct` (cancelled / all orders), `refund_rate_pct` (refunded return amounts on that month's orders / valid revenue) and `revenue_mom_growth_pct`.

**Tables:** `orders`, `order_items`, `returns`

💡 **Hint (Question 250):** Build each KPI block in its own CTE at month grain, join them on month_start and compute ratios in the final SELECT.

In [ ]:
%sql
-- Question 250: write your solution here


# 🎉 Finished?
Update `04_sql_learning_tracker.md`, revisit every question you flagged, and re-solve the Expert questions without hints a week later. Then try rebuilding the Level 9 models as real dbt models in this repository.